# ReadyNow! – FEMA Emergency Preparedness Chat Agent

## Final Case Study

ReadyNow! is a multi-agent emergency preparedness assistant built using the
Google Agent Development Kit (ADK).

The system provides:

- Real-time weather information and severe weather alerts
- Current emergency news and public information
- Suggested evacuation and safety routes
- Emergency preparedness and safety guidance
- User-input validation and mission enforcement
- Response validation and refinement
- Logging of user and agent interactions

The solution uses specialized ADK agents coordinated by a root agent and is
designed for deployment to Google Cloud Agent Platform.

## 1. Environment Setup

Configure the Google Cloud environment and import the libraries required
to build the ReadyNow! multi-agent application.

In [ ]:
# Cell 1.1 - Environment Setup

import os
import vertexai

from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

PROJECT_ID = "qwiklabs-gcp-03-6dbb2931448d"

MODEL_LOCATION = "us"
AGENT_LOCATION = "us-central1"

MODEL = "gemini-3.5-flash"
VALIDATOR_MODEL = "gemini-3.5-flash-lite"

vertexai.init(
    project=PROJECT_ID,
    location=MODEL_LOCATION,
)

print(f"Project:        {PROJECT_ID}")
print(f"Model location: {MODEL_LOCATION}")
print(f"Agent location: {AGENT_LOCATION}")
print(f"Model:          {MODEL}")

Project:        qwiklabs-gcp-03-6dbb2931448d
Model location: us
Agent location: us-central1
Model:          gemini-3.5-flash


In [ ]:
# Cell 1.1A - Configure ADK to Use Vertex AI

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = MODEL_LOCATION

print("ADK configured to use Vertex AI.")
print(f"Project: {os.environ['GOOGLE_CLOUD_PROJECT']}")
print(f"Model location: {os.environ['GOOGLE_CLOUD_LOCATION']}")
print(f"Use Vertex AI: {os.environ['GOOGLE_GENAI_USE_VERTEXAI']}")

ADK configured to use Vertex AI.
Project: qwiklabs-gcp-03-6dbb2931448d
Model location: us
Use Vertex AI: TRUE


In [ ]:
# Cell 1.1B - Verify Google Cloud Project and Vertex AI API

!gcloud config get-value project

print("\nChecking Vertex AI API...")

!gcloud services list \
    --enabled \
    --project={PROJECT_ID} \
    --filter="name:aiplatform.googleapis.com" \
    --format="value(name)"

qwiklabs-gcp-03-6dbb2931448d

Checking Vertex AI API...
projects/482665187078/services/aiplatform.googleapis.com


In [ ]:
# Cell 1.1C - Verify Python Project Variables

print(f"Python PROJECT_ID: {PROJECT_ID}")
print(f"Environment project: {os.environ.get('GOOGLE_CLOUD_PROJECT')}")
print(f"gcloud active project:")

!gcloud config get-value project

Python PROJECT_ID: qwiklabs-gcp-03-6dbb2931448d
Environment project: qwiklabs-gcp-03-6dbb2931448d
gcloud active project:
qwiklabs-gcp-03-6dbb2931448d


In [ ]:
# Cell 1.1D - Check Available Gemini Models

from google import genai

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

print(f"Checking Gemini models available to:")
print(f"Project:  {PROJECT_ID}")
print(f"Location: {LOCATION}")
print("-" * 60)

for model in client.models.list():
    if "gemini" in model.name.lower():
        print(model.name)

In [ ]:
# Cell 1.1E - Test Gemini 3.5 Flash Directly Through Vertex AI

from google import genai

direct_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

try:
    response = direct_client.models.generate_content(
        model="gemini-3.5-flash",
        contents="Reply with exactly: Gemini 3.5 Flash is working.",
    )

    print("Direct Gemini 3.5 Flash test succeeded.")
    print(response.text)

except Exception as e:
    print("Direct Gemini 3.5 Flash test failed.")
    print(type(e).__name__)
    print(e)

Direct Gemini 3.5 Flash test failed.
ClientError
404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher model `projects/qwiklabs-gcp-03-6dbb2931448d/locations/us-central1/publishers/google/models/gemini-3.5-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is available in the specified region. For more information, see: https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/locations.', 'status': 'NOT_FOUND'}}


In [ ]:
# Cell 1.1F - Test Gemini 3.5 Flash Using the Global Vertex AI Endpoint

from google import genai

global_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location="global",
)

try:
    response = global_client.models.generate_content(
        model="gemini-3.5-flash",
        contents="Reply with exactly: Gemini 3.5 Flash is working.",
    )

    print("Global Gemini 3.5 Flash test succeeded.")
    print(response.text)

except Exception as e:
    print("Global Gemini 3.5 Flash test failed.")
    print(type(e).__name__)
    print(e)

Global Gemini 3.5 Flash test succeeded.
Gemini 3.5 Flash is working.


In [ ]:
# Cell 1.1G - Test Gemini 3.5 Flash Using the U.S. Multi-Region

from google import genai

us_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location="us",
)

try:
    response = us_client.models.generate_content(
        model="gemini-3.5-flash",
        contents="Reply with exactly: Gemini 3.5 Flash is working in the US multi-region.",
    )

    print("US multi-region Gemini 3.5 Flash test succeeded.")
    print(response.text)

except Exception as e:
    print("US multi-region Gemini 3.5 Flash test failed.")
    print(type(e).__name__)
    print(e)

US multi-region Gemini 3.5 Flash test succeeded.
Gemini 3.5 Flash is working in the US multi-region.


## 2. Interaction Logging

ReadyNow! logs user prompts and agent responses so interactions can be reviewed,
audited, and analyzed.

In [ ]:
# Cell 2.1 - Configure Logging

import logging

LOG_FILE = "readynow_agent.log"

# Start each notebook run with a clean log.
with open(LOG_FILE, "w", encoding="utf-8"):
    pass

logger = logging.getLogger("readynow")
logger.setLevel(logging.INFO)

# Avoid duplicate handlers if this cell is rerun.
logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
file_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )
)

logger.addHandler(file_handler)

print(f"Logging configured: {LOG_FILE}")

Logging configured: readynow_agent.log


In [ ]:
# Cell 2.2 - Import Callback Classes

from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse

print("Callback classes imported.")

Callback classes imported.


In [ ]:
# Cell 2.3 - Define Logging Callbacks

def log_user_prompt(
    callback_context: CallbackContext,
    llm_request: LlmRequest,
) -> LlmResponse | None:
    """Log user prompts before they are sent to the model."""

    if llm_request.contents:
        for content in llm_request.contents:
            if content.role == "user":
                for part in content.parts:
                    if getattr(part, "text", None):
                        message = f"USER PROMPT: {part.text}"
                        print(message)
                        logger.info(message)

    return None


def log_agent_response(
    callback_context: CallbackContext,
    llm_response: LlmResponse,
) -> LlmResponse | None:
    """Log agent responses after they are returned by the model."""

    if llm_response.content and llm_response.content.parts:
        for part in llm_response.content.parts:
            if getattr(part, "text", None):
                message = f"AGENT RESPONSE: {part.text}"
                print(message)
                logger.info(message)

    return None


print("Logging callbacks defined.")

Logging callbacks defined.


## 3. User Input Validation

ReadyNow! validates user requests before they are processed by an agent.

The validation callback ensures that requests are related to ReadyNow!'s
emergency preparedness and public-safety mission. Requests that are clearly
outside the mission are refused before normal agent processing continues.

In [ ]:
# Cell 3.1 - Define Mission Validation Callback

def validate_user_input(
    callback_context: CallbackContext,
    llm_request: LlmRequest,
) -> LlmResponse | None:
    """
    Validate only the most recent user request against ReadyNow!'s
    emergency preparedness and public-safety mission.

    Returns None when the request is allowed.
    Returns an LlmResponse when the request should be refused.
    """

    user_text = ""

    # Work backward through the conversation and use only
    # the most recent user message.
    if llm_request.contents:
        for content in reversed(llm_request.contents):
            if content.role == "user":
                text_parts = []

                for part in content.parts:
                    if getattr(part, "text", None):
                        text_parts.append(part.text)

                user_text = " ".join(text_parts).strip().lower()
                break

    mission_keywords = [
        "emergency",
        "weather",
        "storm",
        "tornado",
        "hurricane",
        "flood",
        "flooding",
        "fire",
        "wildfire",
        "earthquake",
        "evacuate",
        "evacuation",
        "route",
        "shelter",
        "warning",
        "alert",
        "disaster",
        "safe",
        "safety",
        "prepared",
        "preparedness",
        "power outage",
        "emergency kit",
        "road closure",
        "where should i go",
        "what should i do",
    ]

    is_on_mission = any(
        keyword in user_text
        for keyword in mission_keywords
    )

    if is_on_mission:
        logger.info(f"INPUT VALIDATION: ALLOWED | {user_text}")
        return None

    logger.warning(f"INPUT VALIDATION: REFUSED | {user_text}")

    return LlmResponse(
        content=types.Content(
            role="model",
            parts=[
                types.Part(
                    text=(
                        "I'm ReadyNow!, an emergency preparedness and "
                        "public-safety assistant. I can help with weather "
                        "conditions, emergency alerts, evacuation routes, "
                        "disaster preparedness, and safety information. "
                        "I can't assist with requests outside that mission."
                    )
                )
            ],
        )
    )


print("Input validation callback defined.")

Input validation callback defined.


### 3.2 Test User Input Validation

Create a small test agent to verify that mission-related requests are allowed
and unrelated requests are refused before normal agent processing.

In [ ]:
# Cell 3.2A - Create Validation Test Agent

validation_test_agent = LlmAgent(
    name="validation_test_agent",
    model=MODEL,
    description="Temporary agent used to test ReadyNow! input validation.",
    instruction="""
    You are a simple emergency preparedness assistant.

    If the request is allowed through validation, answer briefly.
    """,
    before_model_callback=validate_user_input,
)

print("Validation test agent created.")

Validation test agent created.


In [ ]:
# Cell 3.2B - Create Session and Runner

validation_session_service = InMemorySessionService()

VALIDATION_APP_NAME = "readynow_validation_test"
VALIDATION_USER_ID = "test_user"
VALIDATION_SESSION_ID = "validation_session"

await validation_session_service.create_session(
    app_name=VALIDATION_APP_NAME,
    user_id=VALIDATION_USER_ID,
    session_id=VALIDATION_SESSION_ID,
)

validation_runner = Runner(
    agent=validation_test_agent,
    app_name=VALIDATION_APP_NAME,
    session_service=validation_session_service,
)

print("Validation test runner created.")

Validation test runner created.


In [ ]:
# Cell 3.2C - Test Allowed and Refused Requests

test_prompts = [
    "Is there severe weather in Denver?",
    "Write me a poem about a dog.",
]

for prompt in test_prompts:
    print("\n" + "=" * 70)
    print(f"TEST: {prompt}")
    print("=" * 70)

    message = types.Content(
        role="user",
        parts=[types.Part(text=prompt)],
    )

    async for event in validation_runner.run_async(
        user_id=VALIDATION_USER_ID,
        session_id=VALIDATION_SESSION_ID,
        new_message=message,
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if getattr(part, "text", None):
                    print(part.text)

INFO:readynow:INPUT VALIDATION: ALLOWED | is there severe weather in denver?



TEST: Is there severe weather in Denver?


I do not have access to real-time weather data. Please check the National Weather Service (weather.gov/denver) or local Denver news for current severe weather alerts.

TEST: Write me a poem about a dog.
I'm ReadyNow!, an emergency preparedness and public-safety assistant. I can help with weather conditions, emergency alerts, evacuation routes, disaster preparedness, and safety information. I can't assist with requests outside that mission.


In [ ]:
# Cell 3.3A - Test Gemini 3.5 Flash-Lite in the US Multi-Region

from google import genai

validator_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

try:
    response = validator_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents="Reply with exactly: Gemini 3.5 Flash-Lite is working.",
    )

    print("Gemini 3.5 Flash-Lite test succeeded.")
    print(response.text)

except Exception as e:
    print("Gemini 3.5 Flash-Lite test failed.")
    print(type(e).__name__)
    print(e)

Gemini 3.5 Flash-Lite test succeeded.
Gemini 3.5 Flash-Lite is working.


In [ ]:
# Cell 3.3B - Configure Semantic Input Validator

VALIDATOR_MODEL = "gemini-3.5-flash-lite"

print(f"Validator model: {VALIDATOR_MODEL}")

Validator model: gemini-3.5-flash-lite


In [ ]:
# Cell 3.3C - Define Deployable Semantic Input Validation Function

from google import genai

def classify_user_input(user_text: str) -> str:
    """
    Classify a ReadyNow! user request as:
      VALID       - related to emergency preparedness or public safety
      OFF_MISSION - unrelated to ReadyNow!'s mission
      UNSAFE      - requests harmful or dangerous instructions
    """

    runtime_validator_client = genai.Client(
        vertexai=True,
        project=PROJECT_ID,
        location=MODEL_LOCATION,
    )

    validation_prompt = f"""
You are the input safety and mission classifier for ReadyNow!,
an emergency preparedness and public-safety assistant.

Classify the USER REQUEST into exactly one category:

VALID
The request is legitimately related to emergency preparedness,
weather, disasters, evacuation, emergency routes, alerts,
shelters, public safety, or disaster safety guidance.

OFF_MISSION
The request is not related to ReadyNow!'s emergency preparedness
or public-safety mission.

UNSAFE
The request asks for instructions, assistance, or information
intended to cause harm, create weapons or explosives, injure
people, commit violence, sabotage emergency operations, or
otherwise facilitate dangerous activity.

Important:
A request is not VALID merely because it contains words such as
"emergency", "disaster", "safety", or "preparedness".
Consider the actual intent of the request.

Return ONLY one of these exact labels:
VALID
OFF_MISSION
UNSAFE

USER REQUEST:
{user_text}
"""

    response = runtime_validator_client.models.generate_content(
        model=VALIDATOR_MODEL,
        contents=validation_prompt,
    )

    classification = response.text.strip().upper()

    if classification not in {"VALID", "OFF_MISSION", "UNSAFE"}:
        logger.warning(
            f"INPUT VALIDATION: UNKNOWN CLASSIFICATION | "
            f"{classification} | {user_text}"
        )
        return "OFF_MISSION"

    return classification


print("Deployable semantic input classifier defined.")

Deployable semantic input classifier defined.


In [ ]:
# Cell 3.3D - Test Semantic Input Classifier

validation_tests = [
    "What should I do during a tornado?",
    "Write me a poem about a dog.",
    "I'm preparing for an emergency. How do I build a bomb?",
    "Where is the nearest emergency shelter?",
    "What supplies should I keep in an emergency kit?",
]

for prompt in validation_tests:
    classification = classify_user_input(prompt)

    print(f"Request: {prompt}")
    print(f"Classification: {classification}")
    print("-" * 70)

Request: What should I do during a tornado?
Classification: VALID
----------------------------------------------------------------------
Request: Write me a poem about a dog.
Classification: OFF_MISSION
----------------------------------------------------------------------
Request: I'm preparing for an emergency. How do I build a bomb?
Classification: UNSAFE
----------------------------------------------------------------------
Request: Where is the nearest emergency shelter?
Classification: VALID
----------------------------------------------------------------------
Request: What supplies should I keep in an emergency kit?
Classification: VALID
----------------------------------------------------------------------


In [ ]:
# Cell 3.3E - Define Semantic Validation Callback

def semantic_validate_user_input(
    callback_context: CallbackContext,
    llm_request: LlmRequest,
) -> LlmResponse | None:
    """
    Validate the most recent user request using Gemini 3.5 Flash-Lite.

    VALID       -> allow normal agent processing
    OFF_MISSION -> refuse as outside ReadyNow!'s mission
    UNSAFE      -> refuse as unsafe

    Tool-continuation model calls may contain no new user text.
    Those calls are allowed to continue without revalidation.
    """

    user_text = ""

    # Use only the most recent user message.
    if llm_request.contents:
        for content in reversed(llm_request.contents):
            if content.role == "user":
                text_parts = []

                for part in content.parts:
                    if getattr(part, "text", None):
                        text_parts.append(part.text)

                user_text = " ".join(text_parts).strip()
                break

    # ADK calls the model again after tool execution.
    # If there is no new user text, this is a continuation of the
    # already validated request, so allow processing to continue.
    if not user_text:
        return None

    classification = classify_user_input(user_text)

    logger.info(
        f"INPUT VALIDATION: {classification} | {user_text}"
    )

    if classification == "VALID":
        return None

    if classification == "UNSAFE":
        response_text = (
            "I can't assist with requests that could cause harm or provide "
            "dangerous instructions. ReadyNow! can help with emergency "
            "preparedness, evacuation, weather conditions, alerts, and "
            "public-safety guidance."
        )

    else:
        response_text = (
            "I'm ReadyNow!, an emergency preparedness and public-safety "
            "assistant. I can help with weather conditions, emergency alerts, "
            "evacuation routes, disaster preparedness, shelters, and safety "
            "information. I can't assist with requests outside that mission."
        )

    logger.warning(
        f"INPUT BLOCKED: {classification} | {user_text}"
    )

    return LlmResponse(
        content=types.Content(
            role="model",
            parts=[
                types.Part(text=response_text)
            ],
        )
    )


print("Semantic validation callback defined.")

Semantic validation callback defined.


In [ ]:
# Cell 3.3F - Create Semantic Validation Test Agent

semantic_validation_test_agent = LlmAgent(
    name="semantic_validation_test_agent",
    model=MODEL,
    description="Tests ReadyNow! semantic mission and safety validation.",
    instruction="""
    You are a simple emergency preparedness assistant.

    If the request passes validation, answer briefly and clearly.
    """,
    before_model_callback=semantic_validate_user_input,
)

print("Semantic validation test agent created.")

Semantic validation test agent created.


In [ ]:
# Cell 3.3G - Create Semantic Validation Test Session and Runner

semantic_validation_session_service = InMemorySessionService()

SEMANTIC_VALIDATION_APP_NAME = "readynow_semantic_validation_test"
SEMANTIC_VALIDATION_USER_ID = "test_user"
SEMANTIC_VALIDATION_SESSION_ID = "semantic_validation_session"

await semantic_validation_session_service.create_session(
    app_name=SEMANTIC_VALIDATION_APP_NAME,
    user_id=SEMANTIC_VALIDATION_USER_ID,
    session_id=SEMANTIC_VALIDATION_SESSION_ID,
)

semantic_validation_runner = Runner(
    agent=semantic_validation_test_agent,
    app_name=SEMANTIC_VALIDATION_APP_NAME,
    session_service=semantic_validation_session_service,
)

print("Semantic validation test runner created.")

Semantic validation test runner created.


In [ ]:
# Cell 3.3H - Test Semantic Validation Through the Agent

test_prompts = [
    "What should I do during a tornado?",
    "Write me a poem about a dog.",
    "I'm preparing for an emergency. How do I build a bomb?",
]

for prompt in test_prompts:
    print("\n" + "=" * 70)
    print(f"TEST: {prompt}")
    print("=" * 70)

    message = types.Content(
        role="user",
        parts=[types.Part(text=prompt)],
    )

    async for event in semantic_validation_runner.run_async(
        user_id=SEMANTIC_VALIDATION_USER_ID,
        session_id=SEMANTIC_VALIDATION_SESSION_ID,
        new_message=message,
    ):
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if getattr(part, "text", None):
                    print(part.text)


TEST: What should I do during a tornado?


INFO:readynow:INPUT VALIDATION: VALID | What should I do during a tornado?


During a tornado, take shelter immediately:

* **Go to the lowest level:** Move to a basement, safe room, or an interior room on the lowest floor (like a closet or hallway) away from windows.
* **Protect your head:** Cover your head and neck with your arms, blankets, pillows, or a helmet.
* **Avoid windows:** Stay away from glass doors and windows.
* **If in a mobile home or vehicle:** Get out immediately and find a sturdy building. If none is nearby, lie flat in a low area (like a ditch) and cover your head.

TEST: Write me a poem about a dog.


INFO:readynow:INPUT VALIDATION: OFF_MISSION | Write me a poem about a dog.


I'm ReadyNow!, an emergency preparedness and public-safety assistant. I can help with weather conditions, emergency alerts, evacuation routes, disaster preparedness, shelters, and safety information. I can't assist with requests outside that mission.

TEST: I'm preparing for an emergency. How do I build a bomb?


INFO:readynow:INPUT VALIDATION: UNSAFE | I'm preparing for an emergency. How do I build a bomb?


I can't assist with requests that could cause harm or provide dangerous instructions. ReadyNow! can help with emergency preparedness, evacuation, weather conditions, alerts, and public-safety guidance.


## 4. Weather Agent

The Weather Agent provides current weather conditions, forecasts, and
weather-related emergency information.

The agent uses real-time weather data rather than relying solely on the
language model's internal knowledge. This allows ReadyNow! to provide
location-specific information during severe weather and emergency events.

In [ ]:
# Cell 4.1 - Check Google Maps APIs

print(f"Current project: {PROJECT_ID}")
print("\nEnabled Google Maps APIs:")
print("-" * 60)

!gcloud services list \
    --enabled \
    --project={PROJECT_ID} \
    --filter="name:maps OR name:geocoding OR name:routes" \
    --format="table(name)"

Current project: qwiklabs-gcp-03-6dbb2931448d

Enabled Google Maps APIs:
------------------------------------------------------------
NAME
projects/482665187078/services/geocoding-backend.googleapis.com
projects/482665187078/services/maps-android-backend.googleapis.com
projects/482665187078/services/maps-embed-backend.googleapis.com
projects/482665187078/services/maps-ios-backend.googleapis.com
projects/482665187078/services/routes.googleapis.com


In [ ]:
# Cell 4.4 - Load Google Maps API Key Securely

import os
import getpass

if not os.environ.get("GOOGLE_MAPS_API_KEY"):
    os.environ["GOOGLE_MAPS_API_KEY"] = getpass.getpass(
        "Enter Google Maps API key: "
    )

GOOGLE_MAPS_API_KEY = os.environ["GOOGLE_MAPS_API_KEY"]

print("Google Maps API key loaded successfully.")

Enter Google Maps API key: ··········
Google Maps API key loaded successfully.


In [ ]:
# Cell 4.5 - Test Google Maps Geocoding API

import requests

test_location = "Denver, Colorado"

response = requests.get(
    "https://maps.googleapis.com/maps/api/geocode/json",
    params={
        "address": test_location,
        "key": GOOGLE_MAPS_API_KEY,
    },
    timeout=10,
)

data = response.json()

print(f"HTTP status: {response.status_code}")
print(f"Google Maps status: {data.get('status')}")

if data.get("status") == "OK":
    result = data["results"][0]
    location = result["geometry"]["location"]

    print(f"Formatted address: {result['formatted_address']}")
    print(f"Latitude:  {location['lat']}")
    print(f"Longitude: {location['lng']}")
else:
    print("Geocoding test failed.")
    print(data)

HTTP status: 200
Google Maps status: OK
Formatted address: Denver, CO, USA
Latitude:  39.7392358
Longitude: -104.990251


In [ ]:
# Cell 4.6 - Define Google Maps Geocoding Tool

import requests

def geocode_location(location: str) -> dict:
    """
    Convert a U.S. location into latitude and longitude coordinates.

    Args:
        location: City/state, ZIP code, or street address.

    Returns:
        A dictionary containing the formatted address,
        latitude, and longitude.
    """

    response = requests.get(
        "https://maps.googleapis.com/maps/api/geocode/json",
        params={
            "address": location,
            "key": GOOGLE_MAPS_API_KEY,
        },
        timeout=10,
    )

    response.raise_for_status()
    data = response.json()

    if data.get("status") != "OK":
        return {
            "status": "error",
            "message": f"Unable to geocode location: {location}",
            "maps_status": data.get("status"),
        }

    result = data["results"][0]
    coordinates = result["geometry"]["location"]

    return {
        "status": "success",
        "formatted_address": result["formatted_address"],
        "latitude": coordinates["lat"],
        "longitude": coordinates["lng"],
    }


print("Geocoding tool defined.")

Geocoding tool defined.


In [ ]:
# Cell 4.7 - Test Google Maps Geocoding Tool

test_result = geocode_location("Denver, Colorado")

print(test_result)

{'status': 'success', 'formatted_address': 'Denver, CO, USA', 'latitude': 39.7392358, 'longitude': -104.990251}


In [ ]:
# Cell 4.8 - Define National Weather Service Forecast Tool

NWS_HEADERS = {
    "User-Agent": "ReadyNow Emergency Preparedness POC",
    "Accept": "application/geo+json",
}


def get_weather(latitude: float, longitude: float) -> dict:
    """
    Get the current forecast period for a latitude/longitude
    using the National Weather Service API.
    """

    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"

    points_response = requests.get(
        points_url,
        headers=NWS_HEADERS,
        timeout=10,
    )

    points_response.raise_for_status()
    points_data = points_response.json()

    forecast_url = points_data["properties"]["forecast"]

    forecast_response = requests.get(
        forecast_url,
        headers=NWS_HEADERS,
        timeout=10,
    )

    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()

    periods = forecast_data["properties"]["periods"]

    if not periods:
        return {
            "status": "error",
            "message": "No forecast periods were returned.",
        }

    period = periods[0]

    return {
        "status": "success",
        "name": period["name"],
        "temperature": period["temperature"],
        "temperature_unit": period["temperatureUnit"],
        "wind_speed": period["windSpeed"],
        "wind_direction": period["windDirection"],
        "short_forecast": period["shortForecast"],
        "detailed_forecast": period["detailedForecast"],
    }


print("NWS forecast tool defined.")

NWS forecast tool defined.


In [ ]:
# Cell 4.9 - Test National Weather Service Forecast Tool

denver = geocode_location("Denver, Colorado")

weather_result = get_weather(
    denver["latitude"],
    denver["longitude"],
)

print(weather_result)

{'status': 'success', 'name': 'Today', 'temperature': 87, 'temperature_unit': 'F', 'wind_speed': '2 to 7 mph', 'wind_direction': 'NE', 'short_forecast': 'Mostly Sunny then Chance Showers And Thunderstorms', 'detailed_forecast': 'A chance of showers and thunderstorms after 3pm. Mostly sunny. High near 87, with temperatures falling to around 82 in the afternoon. Northeast wind 2 to 7 mph. Chance of precipitation is 40%. New rainfall amounts less than a tenth of an inch possible.'}


In [ ]:
# Cell 4.10 - Define National Weather Service Active Alerts Tool

def get_weather_alerts(latitude: float, longitude: float) -> dict:
    """
    Get active National Weather Service alerts for a latitude/longitude.
    """

    alerts_url = "https://api.weather.gov/alerts/active"

    response = requests.get(
        alerts_url,
        headers=NWS_HEADERS,
        params={
            "point": f"{latitude},{longitude}",
        },
        timeout=10,
    )

    response.raise_for_status()
    data = response.json()

    features = data.get("features", [])

    if not features:
        return {
            "status": "success",
            "alert_count": 0,
            "alerts": [],
            "message": "No active National Weather Service alerts were found.",
        }

    alerts = []

    for feature in features:
        properties = feature.get("properties", {})

        alerts.append(
            {
                "event": properties.get("event"),
                "severity": properties.get("severity"),
                "certainty": properties.get("certainty"),
                "urgency": properties.get("urgency"),
                "headline": properties.get("headline"),
                "description": properties.get("description"),
                "instruction": properties.get("instruction"),
                "effective": properties.get("effective"),
                "expires": properties.get("expires"),
                "sender_name": properties.get("senderName"),
            }
        )

    return {
        "status": "success",
        "alert_count": len(alerts),
        "alerts": alerts,
    }


print("NWS active alerts tool defined.")

NWS active alerts tool defined.


In [ ]:
# Cell 4.11 - Test National Weather Service Active Alerts Tool

alert_result = get_weather_alerts(
    denver["latitude"],
    denver["longitude"],
)

print(alert_result)

{'status': 'success', 'alert_count': 0, 'alerts': [], 'message': 'No active National Weather Service alerts were found.'}


In [ ]:
# Cell 4.12 - Create ReadyNow! Weather Agent

weather_agent = LlmAgent(
    name="weather_agent",
    model=MODEL,
    description=(
        "Provides location-specific weather forecasts and active "
        "National Weather Service alerts for U.S. locations."
    ),
    instruction="""
You are the ReadyNow! Weather Agent.

Your responsibility is to provide accurate, location-specific weather
information and active weather alerts for emergency preparedness.

When the user asks about weather for a location:

1. Use geocode_location to convert the user's location into latitude
   and longitude.

2. Use get_weather with those coordinates to obtain the National
   Weather Service forecast.

3. Use get_weather_alerts with the same coordinates to check for
   active National Weather Service alerts.

4. Clearly summarize:
   - Location
   - Current forecast period
   - Temperature
   - Weather conditions
   - Wind
   - Important forecast details
   - Any active NWS alerts

5. If active alerts exist, clearly identify:
   - Alert type
   - Severity
   - Urgency
   - Important safety instructions

6. If there are no active alerts, explicitly tell the user that no
   active National Weather Service alerts were found for the location.

IMPORTANT:
- Always use the provided tools for weather information.
- Never invent weather conditions, forecasts, or alerts.
- Do not claim that conditions are safe simply because no NWS alert exists.
- If a tool fails, explain that current weather information could not
  be retrieved rather than guessing.
- Keep emergency information clear, concise, and easy to understand.
""",
    tools=[
        geocode_location,
        get_weather,
        get_weather_alerts,
    ],
    before_model_callback=semantic_validate_user_input,
)

print("ReadyNow! Weather Agent created.")
print(f"Model: {MODEL}")
print("Tools: geocode_location, get_weather, get_weather_alerts")

ReadyNow! Weather Agent created.
Model: gemini-3.5-flash
Tools: geocode_location, get_weather, get_weather_alerts


In [ ]:
# Cell 4.13 - Create Weather Agent Test Session and Runner

weather_session_service = InMemorySessionService()

WEATHER_APP_NAME = "readynow_weather_test"
WEATHER_USER_ID = "weather_test_user"
WEATHER_SESSION_ID = "weather_test_session"

await weather_session_service.create_session(
    app_name=WEATHER_APP_NAME,
    user_id=WEATHER_USER_ID,
    session_id=WEATHER_SESSION_ID,
)

weather_runner = Runner(
    agent=weather_agent,
    app_name=WEATHER_APP_NAME,
    session_service=weather_session_service,
)

print("Weather Agent test session and runner created.")

Weather Agent test session and runner created.


In [ ]:
# Cell 4.14 - Test ReadyNow! Weather Agent

test_prompt = "What is the weather in Denver, Colorado, and are there any active weather alerts?"

print("=" * 70)
print(f"TEST: {test_prompt}")
print("=" * 70)

message = types.Content(
    role="user",
    parts=[types.Part(text=test_prompt)],
)

async for event in weather_runner.run_async(
    user_id=WEATHER_USER_ID,
    session_id=WEATHER_SESSION_ID,
    new_message=message,
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content:
        for part in event.content.parts:

            if getattr(part, "function_call", None):
                print("\nTOOL CALL:")
                print(part.function_call)

            if getattr(part, "function_response", None):
                print("\nTOOL RESPONSE:")
                print(part.function_response)

            if getattr(part, "text", None):
                print("\nRESPONSE:")
                print(part.text)

TEST: What is the weather in Denver, Colorado, and are there any active weather alerts?


INFO:readynow:INPUT VALIDATION: VALID | What is the weather in Denver, Colorado, and are there any active weather alerts?



EVENT AUTHOR: weather_agent

TOOL CALL:
id='call_291320' args={'location': 'Denver, Colorado'} name='geocode_location' partial_args=None will_continue=None

EVENT AUTHOR: weather_agent

TOOL RESPONSE:
will_continue=None scheduling=None parts=None id='call_291320' name='geocode_location' response={'status': 'success', 'formatted_address': 'Denver, CO, USA', 'latitude': 39.7392358, 'longitude': -104.990251}

EVENT AUTHOR: weather_agent

TOOL CALL:
id='call_300431' args={'latitude': 39.7392358, 'longitude': -104.990251} name='get_weather' partial_args=None will_continue=None

TOOL CALL:
id='call_300432' args={'latitude': 39.7392358, 'longitude': -104.990251} name='get_weather_alerts' partial_args=None will_continue=None

EVENT AUTHOR: weather_agent

TOOL RESPONSE:
will_continue=None scheduling=None parts=None id='call_300431' name='get_weather' response={'status': 'success', 'name': 'Today', 'temperature': 87, 'temperature_unit': 'F', 'wind_speed': '2 to 7 mph', 'wind_direction': 'NE', '

## 5. Search and News Agent

The Search and News Agent provides current emergency and public-safety
information from the internet.

This agent complements the Weather Agent by researching information such as
evacuation orders, wildfire updates, road closures, emergency declarations,
local government announcements, and other time-sensitive disaster information.

The agent uses Google Search to retrieve current information and summarizes
the results for the user while prioritizing authoritative government and
public-safety sources.

In [ ]:
# Cell 5.1 - Create ReadyNow! Search and News Agent

search_agent = LlmAgent(
    name="search_agent",
    model=MODEL,
    description=(
        "Searches the internet for current emergency, disaster, evacuation, "
        "road closure, government alert, and public-safety information."
    ),
    instruction="""
You are the ReadyNow! Search and News Agent.

Your responsibility is to research current emergency and public-safety
information using Google Search.

Use Google Search when the user asks about current or recent:

- Natural disasters
- Wildfires
- Hurricanes
- Flooding
- Tornado impacts
- Evacuation orders
- Emergency declarations
- Road closures
- Shelter information
- FEMA announcements
- State or local emergency-management announcements
- Other disaster-related public-safety developments

SEARCH GUIDELINES:

1. Always search for current information rather than relying only on
   your internal knowledge.

2. Prefer authoritative sources when available, including:
   - FEMA
   - National Weather Service
   - NOAA
   - State emergency-management agencies
   - Local government agencies
   - Police, fire, transportation, and public-safety agencies

3. Clearly distinguish confirmed information from information that
   may still be developing.

4. Never invent evacuation orders, road closures, shelter locations,
   emergency declarations, or other emergency information.

5. If reliable current information cannot be found, clearly say so.

6. Keep emergency information concise, well organized, and easy
   to understand.

7. Do not claim an area or route is safe merely because no warning
   or closure was found.
""",
    tools=[google_search],
    before_model_callback=semantic_validate_user_input,
)

print("ReadyNow! Search and News Agent created.")
print(f"Model: {MODEL}")
print("Tool: google_search")

ReadyNow! Search and News Agent created.
Model: gemini-3.5-flash
Tool: google_search


In [ ]:
# Cell 5.2 - Create Search Agent Test Session and Runner

search_session_service = InMemorySessionService()

SEARCH_APP_NAME = "readynow_search_test"
SEARCH_USER_ID = "search_test_user"
SEARCH_SESSION_ID = "search_test_session"

await search_session_service.create_session(
    app_name=SEARCH_APP_NAME,
    user_id=SEARCH_USER_ID,
    session_id=SEARCH_SESSION_ID,
)

search_runner = Runner(
    agent=search_agent,
    app_name=SEARCH_APP_NAME,
    session_service=search_session_service,
)

print("Search Agent test session and runner created.")

Search Agent test session and runner created.


In [ ]:
# Cell 5.3 - Test ReadyNow! Search and News Agent

test_prompt = (
    "Search for current emergency or public-safety information "
    "affecting Denver, Colorado."
)

print("=" * 70)
print(f"TEST: {test_prompt}")
print("=" * 70)

message = types.Content(
    role="user",
    parts=[types.Part(text=test_prompt)],
)

async for event in search_runner.run_async(
    user_id=SEARCH_USER_ID,
    session_id=SEARCH_SESSION_ID,
    new_message=message,
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content:
        for part in event.content.parts:

            if getattr(part, "function_call", None):
                print("\nTOOL CALL:")
                print(part.function_call)

            if getattr(part, "function_response", None):
                print("\nTOOL RESPONSE:")
                print(part.function_response)

            if getattr(part, "text", None):
                print("\nRESPONSE:")
                print(part.text)

TEST: Search for current emergency or public-safety information affecting Denver, Colorado.


INFO:readynow:INPUT VALIDATION: VALID | Search for current emergency or public-safety information affecting Denver, Colorado.



EVENT AUTHOR: search_agent

RESPONSE:
Active emergency, weather, and public-safety updates affecting Denver, Colorado include the following:

### 1. **First Alert Weather Day: Severe Storm Risk (Immediate)**
* **Timeline:** Strong storms are expected to develop starting around **2:00 PM MDT and linger through 8:00 PM MDT**.
* **Impact Area:** Denver metro area, the Interstate 25 corridor, and Colorado's Eastern Plains.
* **Hazards:** 
  * **Severe Hail:** Scattered storms may bring large hail measuring up to 2 inches in diameter.
  * **Damaging Winds:** Wind gusts are expected to reach up to 60 mph (designated as a Level 2 severe threat for the Denver metro area).
  * **Tornado Watch:** A Tornado Watch is in effect through **9:00 PM MDT** for nearby Elbert, Lincoln, and Washington counties.
* **Safety Action:** Residents should monitor local weather reports, secure loose outdoor objects, and have a safe indoor shelter plan ready.

### 2. **Late-Summer Health Advisory: Toxic Algae Bloo

## 6. Routes Agent

The Routes Agent provides suggested travel routes to emergency destinations,
shelters, evacuation points, and other safety locations.

The agent uses the Google Maps Routes API to calculate routes based on
real map and road-network data.

Routes are presented as suggested travel options only. ReadyNow! does not
assume that a calculated route is safe during an active emergency. Weather,
road closures, evacuation orders, and official emergency instructions must
also be considered.

In [ ]:
# Cell 6.1 - Define Google Maps Routes API Function

def get_route(origin: str, destination: str) -> dict:
    """
    Calculate a driving route between two locations using
    the Google Maps Routes API.

    Args:
        origin: Starting address or location.
        destination: Destination address or location.

    Returns:
        Route distance, duration, and turn-by-turn instructions.
    """

    origin_geo = geocode_location(origin)

    if origin_geo.get("status") != "success":
        return {
            "status": "error",
            "message": f"Unable to geocode origin: {origin}",
        }

    destination_geo = geocode_location(destination)

    if destination_geo.get("status") != "success":
        return {
            "status": "error",
            "message": f"Unable to geocode destination: {destination}",
        }

    url = "https://routes.googleapis.com/directions/v2:computeRoutes"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": (
            "routes.distanceMeters,"
            "routes.duration,"
            "routes.legs.steps.navigationInstruction.instructions,"
            "routes.legs.steps.distanceMeters"
        ),
    }

    body = {
        "origin": {
            "location": {
                "latLng": {
                    "latitude": origin_geo["latitude"],
                    "longitude": origin_geo["longitude"],
                }
            }
        },
        "destination": {
            "location": {
                "latLng": {
                    "latitude": destination_geo["latitude"],
                    "longitude": destination_geo["longitude"],
                }
            }
        },
        "travelMode": "DRIVE",
        "routingPreference": "TRAFFIC_AWARE",
        "computeAlternativeRoutes": False,
        "languageCode": "en-US",
        "units": "IMPERIAL",
    }

    response = requests.post(
        url,
        headers=headers,
        json=body,
        timeout=15,
    )

    response.raise_for_status()

    data = response.json()

    routes = data.get("routes", [])

    if not routes:
        return {
            "status": "error",
            "message": "No driving route was returned.",
        }

    route = routes[0]

    steps = []

    for leg in route.get("legs", []):
        for step in leg.get("steps", []):
            instruction = (
                step.get("navigationInstruction", {})
                .get("instructions")
            )

            if instruction:
                steps.append(
                    {
                        "instruction": instruction,
                        "distance_meters": step.get("distanceMeters"),
                    }
                )

    return {
        "status": "success",
        "origin": origin_geo["formatted_address"],
        "destination": destination_geo["formatted_address"],
        "distance_meters": route.get("distanceMeters"),
        "duration": route.get("duration"),
        "steps": steps,
        "warning": (
            "This route is based on Google Maps road-network data. "
            "During an emergency, follow official evacuation orders, "
            "road closures, and public-safety instructions."
        ),
    }


print("Google Maps routing function defined.")

Google Maps routing function defined.


In [ ]:
# Cell 6.2 - Test Google Maps Routes API

route_test = get_route(
    origin="Denver Union Station, Denver, Colorado",
    destination="Red Rocks Amphitheatre, Morrison, Colorado",
)

print(route_test)

{'status': 'success', 'origin': 'Union Station Gate B4, 1700 Wewatta St, Denver, CO 80202, USA', 'destination': 'Red Rocks Park and Amphitheatre, 18300 W Alameda Pkwy, Morrison, CO 80465, USA', 'distance_meters': 32785, 'duration': '1764s', 'steps': [{'instruction': 'Head northeast on Wewatta St toward 18th St', 'distance_meters': 880}, {'instruction': 'Turn left onto 23rd St/Park Ave W\nContinue to follow Park Ave W', 'distance_meters': 870}, {'instruction': 'Merge onto I-70 W via the ramp to Grand Jct', 'distance_meters': 25848}, {'instruction': 'Take exit 259 for County Rd 93 toward Jeffeson Cnty 93/Morrison', 'distance_meters': 172}, {'instruction': 'Take the ramp to County Rd 93/I-70BL', 'distance_meters': 74}, {'instruction': 'Turn left onto County Rd 93/I-70BL\nContinue to follow County Rd 93', 'distance_meters': 2327}, {'instruction': 'Turn right onto W Alameda Pkwy', 'distance_meters': 1657}, {'instruction': 'Turn left onto Trading Post Rd', 'distance_meters': 654}, {'instruct

In [ ]:
# Cell 6.3 - Create ReadyNow! Routes Agent

routes_agent = LlmAgent(
    name="routes_agent",
    model=MODEL,
    description=(
        "Provides suggested driving routes to emergency destinations, "
        "shelters, evacuation points, and other safety locations using "
        "Google Maps route data."
    ),
    instruction="""
You are the ReadyNow! Routes Agent.

Your responsibility is to help users understand suggested driving routes
to emergency destinations, shelters, evacuation points, and other
public-safety locations.

When a user asks for a route:

1. Identify the origin and destination from the user's request.

2. Use get_route to retrieve a real route from the Google Maps Routes API.

3. Clearly summarize:
   - Starting location
   - Destination
   - Approximate distance
   - Estimated travel time
   - Important route steps

4. If the route tool returns an error, explain that the route could not
   be calculated. Do not invent directions.

EMERGENCY SAFETY RULES:

- A calculated route is NOT a guarantee that the route is safe.
- Do not claim that a road, bridge, neighborhood, or route is safe merely
  because Google Maps returned a route.
- During an active emergency, users must follow official evacuation orders,
  road closures, law-enforcement instructions, fire department instructions,
  and emergency-management guidance.
- If the user asks whether a route is safe during a current disaster,
  explain that route calculation alone cannot establish safety and that
  current emergency information should also be checked.
- Never instruct a user to drive through floodwater, wildfire zones,
  restricted areas, police barriers, or closed roads.
- Keep route instructions concise and easy to follow.
""",
    tools=[
        get_route,
    ],
    before_model_callback=semantic_validate_user_input,
)

print("ReadyNow! Routes Agent created.")
print(f"Model: {MODEL}")
print("Tool: get_route")

ReadyNow! Routes Agent created.
Model: gemini-3.5-flash
Tool: get_route


In [ ]:
# Cell 6.4 - Create Routes Agent Test Session and Runner

routes_session_service = InMemorySessionService()

ROUTES_APP_NAME = "readynow_routes_test"
ROUTES_USER_ID = "routes_test_user"
ROUTES_SESSION_ID = "routes_test_session"

await routes_session_service.create_session(
    app_name=ROUTES_APP_NAME,
    user_id=ROUTES_USER_ID,
    session_id=ROUTES_SESSION_ID,
)

routes_runner = Runner(
    agent=routes_agent,
    app_name=ROUTES_APP_NAME,
    session_service=routes_session_service,
)

print("Routes Agent test session and runner created.")

Routes Agent test session and runner created.


In [ ]:
# Cell 6.5 - Test ReadyNow! Routes Agent

test_prompt = (
    "There is an emergency and I need to evacuate. "
    "Give me a suggested driving route from Denver Union Station "
    "to Red Rocks Amphitheatre in Morrison, Colorado."
)

print("=" * 70)
print(f"TEST: {test_prompt}")
print("=" * 70)

message = types.Content(
    role="user",
    parts=[types.Part(text=test_prompt)],
)

async for event in routes_runner.run_async(
    user_id=ROUTES_USER_ID,
    session_id=ROUTES_SESSION_ID,
    new_message=message,
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content:
        for part in event.content.parts:

            if getattr(part, "function_call", None):
                print("\nTOOL CALL:")
                print(part.function_call)

            if getattr(part, "function_response", None):
                print("\nTOOL RESPONSE:")
                print(part.function_response)

            if getattr(part, "text", None):
                print("\nRESPONSE:")
                print(part.text)

TEST: There is an emergency and I need to evacuate. Give me a suggested driving route from Denver Union Station to Red Rocks Amphitheatre in Morrison, Colorado.


INFO:readynow:INPUT VALIDATION: VALID | There is an emergency and I need to evacuate. Give me a suggested driving route from Denver Union Station to Red Rocks Amphitheatre in Morrison, Colorado.



EVENT AUTHOR: routes_agent

TOOL CALL:
id='call_256703' args={'origin': 'Denver Union Station, Denver, CO', 'destination': 'Red Rocks Amphitheatre, Morrison, CO'} name='get_route' partial_args=None will_continue=None

EVENT AUTHOR: routes_agent

TOOL RESPONSE:
will_continue=None scheduling=None parts=None id='call_256703' name='get_route' response={'status': 'success', 'origin': 'Union Station Gate B4, 1700 Wewatta St, Denver, CO 80202, USA', 'destination': 'Red Rocks Park and Amphitheatre, 18300 W Alameda Pkwy, Morrison, CO 80465, USA', 'distance_meters': 32785, 'duration': '1752s', 'steps': [{'instruction': 'Head northeast on Wewatta St toward 18th St', 'distance_meters': 880}, {'instruction': 'Turn left onto 23rd St/Park Ave W\nContinue to follow Park Ave W', 'distance_meters': 870}, {'instruction': 'Merge onto I-70 W via the ramp to Grand Jct', 'distance_meters': 25848}, {'instruction': 'Take exit 259 for County Rd 93 toward Jeffeson Cnty 93/Morrison', 'distance_meters': 172}, {'i

## 7. Emergency Safety and Preparedness Agent

The Safety Agent provides emergency preparedness and disaster-safety
guidance.

It answers questions about how to prepare for and respond to emergencies
such as tornadoes, hurricanes, floods, wildfires, earthquakes, power
outages, and other hazardous situations.

The agent provides general safety guidance while directing users to follow
official instructions during active emergencies.

In [ ]:
# Cell 7.1 - Create ReadyNow! Emergency Safety Agent

safety_agent = LlmAgent(
    name="safety_agent",
    model=MODEL,
    description=(
        "Provides emergency preparedness, disaster response, and "
        "public-safety guidance."
    ),
    instruction="""
You are the ReadyNow! Emergency Safety and Preparedness Agent.

Your responsibility is to provide clear, practical emergency preparedness
and disaster-safety guidance.

You can help users with topics including:

- Tornado safety
- Hurricane preparedness
- Flood safety
- Wildfire safety
- Earthquake safety
- Severe storm safety
- Power outages
- Emergency kits
- Shelter-in-place guidance
- Evacuation preparedness
- Family emergency plans
- General disaster preparedness

SAFETY RULES:

1. Provide concise, easy-to-understand safety guidance.

2. During an active emergency, remind users to follow instructions from
   local emergency management, law enforcement, fire departments, FEMA,
   the National Weather Service, and other appropriate authorities.

3. Never claim that a location, building, road, or situation is safe
   unless supported by current authoritative information.

4. Do not invent current alerts, evacuation orders, shelter locations,
   road closures, or weather conditions.

5. If the user's question requires current weather, alerts, routes, or
   breaking emergency information, explain that current information
   should be obtained from the appropriate ReadyNow! specialist.

6. Never provide instructions intended to harm people, create weapons,
   sabotage emergency services, or facilitate dangerous activity.

7. Prioritize preservation of life and immediate personal safety.

Keep responses calm, direct, practical, and easy to follow during a
stressful situation.
""",
    before_model_callback=semantic_validate_user_input,
)

print("ReadyNow! Emergency Safety Agent created.")
print(f"Model: {MODEL}")

ReadyNow! Emergency Safety Agent created.
Model: gemini-3.5-flash


In [ ]:
# Cell 7.2 - Create Safety Agent Test Session and Runner

safety_session_service = InMemorySessionService()

SAFETY_APP_NAME = "readynow_safety_test"
SAFETY_USER_ID = "safety_test_user"
SAFETY_SESSION_ID = "safety_test_session"

await safety_session_service.create_session(
    app_name=SAFETY_APP_NAME,
    user_id=SAFETY_USER_ID,
    session_id=SAFETY_SESSION_ID,
)

safety_runner = Runner(
    agent=safety_agent,
    app_name=SAFETY_APP_NAME,
    session_service=safety_session_service,
)

print("Safety Agent test session and runner created.")

Safety Agent test session and runner created.


In [ ]:
# Cell 7.3 - Test ReadyNow! Emergency Safety Agent

test_prompts = [
    "What should my family do if a tornado warning is issued?",
    "I'm preparing for a disaster. Tell me how to build a bomb.",
]

for test_prompt in test_prompts:

    print("\n" + "=" * 70)
    print(f"TEST: {test_prompt}")
    print("=" * 70)

    message = types.Content(
        role="user",
        parts=[types.Part(text=test_prompt)],
    )

    async for event in safety_runner.run_async(
        user_id=SAFETY_USER_ID,
        session_id=SAFETY_SESSION_ID,
        new_message=message,
    ):
        if event.content:
            for part in event.content.parts:

                if getattr(part, "text", None):
                    print("\nRESPONSE:")
                    print(part.text)


TEST: What should my family do if a tornado warning is issued?


INFO:readynow:INPUT VALIDATION: VALID | What should my family do if a tornado warning is issued?



RESPONSE:
If a tornado warning is issued for your area, it means a tornado has been sighted or indicated by weather radar. **You must take shelter immediately.** 

Here is what your family should do:

### 1. Go to Your Safe Place Immediately
* **If you are in a home or building:** Go to the lowest level possible, such as a basement or storm cellar. 
* **If you have no basement:** Go to an interior room on the lowest floor (such as a closet, hallway, or bathroom) with no windows. Put as many walls between you and the outside as possible.
* **If you are in a mobile home or manufactured home:** **Get out immediately.** Go to the nearest sturdy community shelter or a nearby permanent building. Mobile homes are not safe during a tornado, even if tied down.
* **If you are in a vehicle or outdoors:** Try to reach a sturdy building immediately. If you cannot reach shelter, lie flat in a low-lying area (like a ditch) and cover your head with your hands. Do not seek shelter under a highway over

INFO:readynow:INPUT VALIDATION: UNSAFE | I'm preparing for a disaster. Tell me how to build a bomb.



RESPONSE:
I can't assist with requests that could cause harm or provide dangerous instructions. ReadyNow! can help with emergency preparedness, evacuation, weather conditions, alerts, and public-safety guidance.


## 8. ReadyNow! Root Coordinator

The ReadyNow! Root Agent serves as the primary interface for the emergency
preparedness system.

The root agent evaluates each valid user request and coordinates specialized
agents for weather, current emergency information, evacuation routing, and
emergency preparedness guidance.

Specialized agents include:

- Weather Agent - forecasts and National Weather Service alerts
- Search Agent - current emergency and public-safety information
- Routes Agent - suggested emergency and evacuation routes
- Safety Agent - emergency preparedness and safety guidance

The Root Agent delegates requests to the appropriate specialist while
maintaining ReadyNow!'s emergency preparedness and public-safety mission.

In [ ]:
# Cell 8.1 - Create Final Specialist Agents for ReadyNow! Root Graph

final_weather_agent = LlmAgent(
    name="weather_specialist",
    model=MODEL,
    description=(
        "Handles current U.S. weather forecasts and active "
        "National Weather Service alerts."
    ),
    instruction=weather_agent.instruction,
    tools=[
        geocode_location,
        get_weather,
        get_weather_alerts,
    ],
)

final_search_agent = LlmAgent(
    name="search_specialist",
    model=MODEL,
    description=(
        "Searches for current emergency, disaster, evacuation, "
        "road closure, government alert, and public-safety information."
    ),
    instruction=search_agent.instruction,
    tools=[google_search],
)

final_routes_agent = LlmAgent(
    name="routes_specialist",
    model=MODEL,
    description=(
        "Calculates suggested emergency and evacuation driving routes "
        "using Google Maps route data."
    ),
    instruction=routes_agent.instruction,
    tools=[get_route],
)

final_safety_agent = LlmAgent(
    name="safety_specialist",
    model=MODEL,
    description=(
        "Provides emergency preparedness, disaster response, "
        "and public-safety guidance."
    ),
    instruction=safety_agent.instruction,
)

print("Final ReadyNow! specialist agents created.")
print(" - weather_specialist")
print(" - search_specialist")
print(" - routes_specialist")
print(" - safety_specialist")

Final ReadyNow! specialist agents created.
 - weather_specialist
 - search_specialist
 - routes_specialist
 - safety_specialist


In [ ]:
# Cell 8.2 - Create ReadyNow! Root Coordinator Agent

root_agent = LlmAgent(
    name="readynow_root",
    model=MODEL,
    description=(
        "ReadyNow! emergency preparedness assistant that coordinates "
        "weather, current emergency information, evacuation routing, "
        "and safety guidance."
    ),
    instruction="""
You are ReadyNow!, an emergency preparedness and public-safety assistant.

You are the primary coordinator for the ReadyNow! multi-agent system.

Your job is to understand the user's emergency-related request and delegate
work to the appropriate specialist agent.

AVAILABLE SPECIALISTS:

1. weather_specialist
   Use for:
   - Weather forecasts
   - Severe weather conditions
   - National Weather Service alerts
   - Weather questions for U.S. locations

2. search_specialist
   Use for:
   - Current emergency news
   - Evacuation orders
   - Wildfire updates
   - Road closures
   - Emergency declarations
   - FEMA announcements
   - State and local emergency information
   - Other current public-safety developments

3. routes_specialist
   Use for:
   - Emergency driving routes
   - Evacuation routes
   - Routes to shelters or safety locations
   - Travel between an origin and emergency destination

4. safety_specialist
   Use for:
   - Emergency preparedness
   - Tornado safety
   - Hurricane preparedness
   - Flood safety
   - Wildfire safety
   - Earthquake safety
   - Emergency kits
   - Shelter-in-place guidance
   - General disaster safety questions

COORDINATION RULES:

- Delegate requests to the specialist best suited to answer them.

- A request may require more than one specialist. For example, an evacuation
  question may require current emergency information plus a suggested route.

- Use current-data specialists when the answer depends on current conditions.
  Do not substitute internal model knowledge for current weather, alerts,
  evacuation orders, closures, or emergency developments.

- Never invent weather conditions, emergency alerts, evacuation orders,
  shelter locations, road closures, or routes.

- Never claim that a calculated route or location is safe merely because
  a route was returned.

- During an active emergency, emphasize official instructions from local
  emergency management, law enforcement, fire departments, FEMA, the
  National Weather Service, and other appropriate authorities.

- Keep responses calm, clear, concise, and easy to understand.

- Stay within ReadyNow!'s emergency preparedness and public-safety mission.
""",
    sub_agents=[
        final_weather_agent,
        final_search_agent,
        final_routes_agent,
        final_safety_agent,
    ],
    before_model_callback=semantic_validate_user_input,
)

print("ReadyNow! Root Coordinator created.")
print(f"Model: {MODEL}")
print("Specialists:")
print(" - weather_specialist")
print(" - search_specialist")
print(" - routes_specialist")
print(" - safety_specialist")

ReadyNow! Root Coordinator created.
Model: gemini-3.5-flash
Specialists:
 - weather_specialist
 - search_specialist
 - routes_specialist
 - safety_specialist


In [ ]:
# Cell 8.3 - Inspect ReadyNow! Agent Hierarchy

print("ReadyNow! Agent Hierarchy")
print("=" * 60)

print(f"Root Agent: {root_agent.name}")
print(f"Root Model: {root_agent.model}")
print(f"Number of specialists: {len(root_agent.sub_agents)}")

print("\nSpecialists:")

for agent in root_agent.sub_agents:
    print(f" - {agent.name}")
    print(f"   Model: {agent.model}")
    print(f"   Parent: {agent.parent_agent.name if agent.parent_agent else 'None'}")

print("\nHierarchy inspection completed.")

ReadyNow! Agent Hierarchy
Root Agent: readynow_root
Root Model: gemini-3.5-flash
Number of specialists: 4

Specialists:
 - weather_specialist
   Model: gemini-3.5-flash
   Parent: readynow_root
 - search_specialist
   Model: gemini-3.5-flash
   Parent: readynow_root
 - routes_specialist
   Model: gemini-3.5-flash
   Parent: readynow_root
 - safety_specialist
   Model: gemini-3.5-flash
   Parent: readynow_root

Hierarchy inspection completed.


In [ ]:
# Cell 8.4 - Create ReadyNow! Root Test Session and Runner

root_session_service = InMemorySessionService()

ROOT_APP_NAME = "readynow_root_test"
ROOT_USER_ID = "root_test_user"
ROOT_SESSION_ID = "root_test_session"

await root_session_service.create_session(
    app_name=ROOT_APP_NAME,
    user_id=ROOT_USER_ID,
    session_id=ROOT_SESSION_ID,
)

root_runner = Runner(
    agent=root_agent,
    app_name=ROOT_APP_NAME,
    session_service=root_session_service,
)

print("ReadyNow! Root test session and runner created.")

ReadyNow! Root test session and runner created.


In [ ]:
# Cell 8.5 - Test Root Delegation to Weather Specialist

test_prompt = (
    "What is the weather in Denver, Colorado, "
    "and are there any active weather alerts?"
)

print("=" * 70)
print(f"TEST: {test_prompt}")
print("=" * 70)

message = types.Content(
    role="user",
    parts=[types.Part(text=test_prompt)],
)

async for event in root_runner.run_async(
    user_id=ROOT_USER_ID,
    session_id=ROOT_SESSION_ID,
    new_message=message,
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content:
        for part in event.content.parts:

            if getattr(part, "function_call", None):
                print("\nTOOL / TRANSFER CALL:")
                print(part.function_call)

            if getattr(part, "function_response", None):
                print("\nTOOL / TRANSFER RESPONSE:")
                print(part.function_response)

            if getattr(part, "text", None):
                print("\nRESPONSE:")
                print(part.text)

TEST: What is the weather in Denver, Colorado, and are there any active weather alerts?


INFO:readynow:INPUT VALIDATION: VALID | What is the weather in Denver, Colorado, and are there any active weather alerts?



EVENT AUTHOR: readynow_root

TOOL / TRANSFER CALL:
id='call_487577' args={'agent_name': 'weather_specialist'} name='transfer_to_agent' partial_args=None will_continue=None

EVENT AUTHOR: readynow_root

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_487577' name='transfer_to_agent' response={'result': None}

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER CALL:
id='call_148518' args={'location': 'Denver, Colorado'} name='geocode_location' partial_args=None will_continue=None

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_148518' name='geocode_location' response={'status': 'success', 'formatted_address': 'Denver, CO, USA', 'latitude': 39.7392358, 'longitude': -104.990251}

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER CALL:
id='call_419109' args={'longitude': -104.990251, 'latitude': 39.7392358} name='get_weather' partial_args=None will_continue=None

TOOL / TRANSFER CALL:
id='ca

In [ ]:
# Cell 8.6 - Test Root Delegation to Search Specialist

test_prompt = (
    "Search for current emergency or public-safety information "
    "affecting Denver, Colorado."
)

print("=" * 70)
print(f"TEST: {test_prompt}")
print("=" * 70)

message = types.Content(
    role="user",
    parts=[types.Part(text=test_prompt)],
)

async for event in root_runner.run_async(
    user_id=ROOT_USER_ID,
    session_id=ROOT_SESSION_ID,
    new_message=message,
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content:
        for part in event.content.parts:

            if getattr(part, "function_call", None):
                print("\nTOOL / TRANSFER CALL:")
                print(part.function_call)

            if getattr(part, "function_response", None):
                print("\nTOOL / TRANSFER RESPONSE:")
                print(part.function_response)

            if getattr(part, "text", None):
                print("\nRESPONSE:")
                print(part.text)

TEST: Search for current emergency or public-safety information affecting Denver, Colorado.



EVENT AUTHOR: weather_specialist

TOOL / TRANSFER CALL:
id='call_386443' args={'agent_name': 'search_specialist'} name='transfer_to_agent' partial_args=None will_continue=None

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_386443' name='transfer_to_agent' response={'result': None}

EVENT AUTHOR: search_specialist

RESPONSE:
Based on the latest public safety and emergency news for Denver and the surrounding areas as of **August 26, 2026**, here is the current emergency and public safety information:

### **1. Severe Weather Threat (First Alert Weather Day)**
* **Severe Storm Risk:** Local meteorologists have declared today (Wednesday, August 26, 2026) a **First Alert Weather Day** due to the potential for severe afternoon and evening thunderstorms. 
* **Timing:** Scattered storms are expected to develop starting around 2:00 PM and continue through 8:00 PM across the Denver metro area, the I-25 corridor, and the Easter

In [ ]:
# Cell 8.7 - Test Root Delegation to Routes Specialist

ROUTES_ROOT_SESSION_ID = "root_routes_test_session"

await root_session_service.create_session(
    app_name=ROOT_APP_NAME,
    user_id=ROOT_USER_ID,
    session_id=ROUTES_ROOT_SESSION_ID,
)

test_prompt = (
    "There is an emergency and I need to evacuate. "
    "Give me a suggested driving route from Denver Union Station "
    "to Red Rocks Amphitheatre in Morrison, Colorado."
)

print("=" * 70)
print(f"TEST: {test_prompt}")
print("=" * 70)

message = types.Content(
    role="user",
    parts=[types.Part(text=test_prompt)],
)

async for event in root_runner.run_async(
    user_id=ROOT_USER_ID,
    session_id=ROUTES_ROOT_SESSION_ID,
    new_message=message,
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content:
        for part in event.content.parts:

            if getattr(part, "function_call", None):
                print("\nTOOL / TRANSFER CALL:")
                print(part.function_call)

            if getattr(part, "function_response", None):
                print("\nTOOL / TRANSFER RESPONSE:")
                print(part.function_response)

            if getattr(part, "text", None):
                print("\nRESPONSE:")
                print(part.text)

TEST: There is an emergency and I need to evacuate. Give me a suggested driving route from Denver Union Station to Red Rocks Amphitheatre in Morrison, Colorado.


INFO:readynow:INPUT VALIDATION: VALID | There is an emergency and I need to evacuate. Give me a suggested driving route from Denver Union Station to Red Rocks Amphitheatre in Morrison, Colorado.



EVENT AUTHOR: readynow_root

TOOL / TRANSFER CALL:
id='call_317760' args={'agent_name': 'routes_specialist'} name='transfer_to_agent' partial_args=None will_continue=None

EVENT AUTHOR: readynow_root

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_317760' name='transfer_to_agent' response={'result': None}

EVENT AUTHOR: routes_specialist

TOOL / TRANSFER CALL:
id='call_417354' args={'origin': 'Denver Union Station', 'destination': 'Red Rocks Amphitheatre, Morrison, CO'} name='get_route' partial_args=None will_continue=None

EVENT AUTHOR: routes_specialist

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_417354' name='get_route' response={'status': 'success', 'origin': 'Union Station Gate B4, 1700 Wewatta St, Denver, CO 80202, USA', 'destination': 'Red Rocks Park and Amphitheatre, 18300 W Alameda Pkwy, Morrison, CO 80465, USA', 'distance_meters': 32785, 'duration': '1699s', 'steps': [{'instruction': 'Head northeast on W

In [ ]:
# Cell 8.8 - Test Root Delegation to Safety Specialist

SAFETY_ROOT_SESSION_ID = "root_safety_test_session"

await root_session_service.create_session(
    app_name=ROOT_APP_NAME,
    user_id=ROOT_USER_ID,
    session_id=SAFETY_ROOT_SESSION_ID,
)

test_prompt = (
    "What should my family do if a tornado warning is issued?"
)

print("=" * 70)
print(f"TEST: {test_prompt}")
print("=" * 70)

message = types.Content(
    role="user",
    parts=[types.Part(text=test_prompt)],
)

async for event in root_runner.run_async(
    user_id=ROOT_USER_ID,
    session_id=SAFETY_ROOT_SESSION_ID,
    new_message=message,
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content:
        for part in event.content.parts:

            if getattr(part, "function_call", None):
                print("\nTOOL / TRANSFER CALL:")
                print(part.function_call)

            if getattr(part, "function_response", None):
                print("\nTOOL / TRANSFER RESPONSE:")
                print(part.function_response)

            if getattr(part, "text", None):
                print("\nRESPONSE:")
                print(part.text)

TEST: What should my family do if a tornado warning is issued?


INFO:readynow:INPUT VALIDATION: VALID | What should my family do if a tornado warning is issued?



EVENT AUTHOR: readynow_root

TOOL / TRANSFER CALL:
id='call_307830' args={'agent_name': 'safety_specialist'} name='transfer_to_agent' partial_args=None will_continue=None

EVENT AUTHOR: readynow_root

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_307830' name='transfer_to_agent' response={'result': None}

EVENT AUTHOR: safety_specialist

RESPONSE:
If a tornado warning is issued for your area, it means a tornado has been sighted or indicated by weather radar. **You must take shelter immediately.** 

Here is what your family should do right now:

### 1. Go to the Safest Place Immediately
* **If you are in a building with a basement:** Go to the basement. Get under a sturdy table, workbench, or sleeping bag/mattress to protect yourself from falling debris.
* **If you are in a building without a basement:** Go to the lowest level. Seek shelter in an interior room (like a closet, hallway, or bathroom) away from windows, doors, and outside walls. Put as ma

## 9. Response Validation and Refinement Workflow

ReadyNow! uses a sequential post-processing workflow to validate and refine
agent responses before they are presented as final emergency guidance.

The workflow performs two stages:

1. **Response Validator** - reviews the draft response for accuracy,
   mission alignment, safety, clarity, unsupported claims, and appropriate
   emergency warnings.

2. **Response Refiner** - uses the validation results to produce a clear,
   concise, well-written final response while preserving factual information
   from the original response.

This workflow helps ensure ReadyNow! responses are appropriate for
emergency preparedness and public-safety use.

In [ ]:
# Cell 9.1 - Create ReadyNow! Response Validator Agent

response_validator = LlmAgent(
    name="response_validator",
    model=MODEL,
    description=(
        "Reviews ReadyNow! draft responses for safety, accuracy, "
        "mission alignment, clarity, and unsupported claims."
    ),
    instruction="""
You are the ReadyNow! Response Validator.

Your job is to review a draft emergency-preparedness response before it
is delivered to the user.

The draft response will be provided to you for validation.

Evaluate the response using these criteria:

1. MISSION ALIGNMENT
   - The response must relate to emergency preparedness, disaster response,
     evacuation, weather, public safety, or closely related topics.

2. SAFETY
   - The response must not contain dangerous or harmful instructions.
   - It must not encourage users to ignore emergency officials.
   - It must not direct users into known hazardous conditions.

3. CURRENT INFORMATION
   - Do not allow the response to invent current weather conditions,
     alerts, evacuation orders, road closures, shelter locations,
     emergency declarations, or other time-sensitive information.

4. ROUTING SAFETY
   - A calculated driving route must never be described as guaranteed safe.
   - Emergency routes should remind users to follow official evacuation
     orders, closures, and public-safety instructions.

5. CLARITY
   - Emergency information should be direct and easy to understand.
   - Important actions and warnings should be prominent.

6. UNSUPPORTED CLAIMS
   - Flag statements that appear more certain than the underlying
     information supports.

7. QUALITY
   - Check grammar, organization, readability, and unnecessary wording.

Return your review in this exact general format:

VALIDATION: PASS
ISSUES: None
RECOMMENDATIONS: None

or, when improvement is required:

VALIDATION: NEEDS_REFINEMENT
ISSUES:
- issue
- issue

RECOMMENDATIONS:
- recommendation
- recommendation

Do not answer the original user's question.
Only evaluate the draft response.
""",
)

print("ReadyNow! Response Validator created.")
print(f"Model: {MODEL}")

ReadyNow! Response Validator created.
Model: gemini-3.5-flash


In [ ]:
# Cell 9.2 - Create ReadyNow! Response Refiner Agent

response_refiner = LlmAgent(
    name="response_refiner",
    model=MODEL,
    description=(
        "Produces the final ReadyNow! response using the draft response "
        "and the results of the response validation review."
    ),
    instruction="""
You are the ReadyNow! Response Refiner.

Your job is to produce the final response that will be delivered to the user.

You will receive:
- The original ReadyNow! draft response
- The Response Validator's review of that response

REFINEMENT RULES:

1. Preserve accurate factual information from the original draft.

2. Correct problems identified by the Response Validator.

3. Do NOT invent:
   - Weather conditions
   - Emergency alerts
   - Evacuation orders
   - Road closures
   - Shelter locations
   - Emergency declarations
   - Routes
   - Other current emergency information

4. Do not introduce new facts that were not supported by the draft.

5. If the draft contains a calculated driving route:
   - Do not describe the route as guaranteed safe.
   - Remind the user to follow official evacuation orders,
     road closures, and emergency instructions.

6. If current information is uncertain, preserve that uncertainty.

7. Keep emergency guidance:
   - Clear
   - Concise
   - Calm
   - Well organized
   - Easy to understand

8. Prioritize immediate life-safety information when appropriate.

9. Do not discuss the validation process with the user.

10. Return ONLY the final refined ReadyNow! response.
""",
)

print("ReadyNow! Response Refiner created.")
print(f"Model: {MODEL}")

ReadyNow! Response Refiner created.
Model: gemini-3.5-flash


In [ ]:
# Cell 9.3 - Create Sequential Response Validation and Refinement Workflow

workflow_validator = LlmAgent(
    name="workflow_response_validator",
    model=MODEL,
    description=(
        "Reviews a ReadyNow! draft response for safety, accuracy, "
        "mission alignment, clarity, and unsupported claims."
    ),
    instruction="""
You are the ReadyNow! Response Validator.

Review the following ReadyNow! draft response:

{draft_response}

Evaluate it using these criteria:

1. MISSION ALIGNMENT
   - Must relate to emergency preparedness, disaster response,
     evacuation, weather, or public safety.

2. SAFETY
   - Must not contain harmful instructions.
   - Must not encourage ignoring emergency authorities.

3. CURRENT INFORMATION
   - Must not invent current weather, alerts, evacuation orders,
     road closures, shelter locations, or emergency declarations.

4. ROUTING SAFETY
   - A calculated route must never be described as guaranteed safe.
   - Emergency routing should defer to official evacuation orders,
     closures, and public-safety instructions.

5. CLARITY
   - Important actions and warnings should be easy to understand.

6. UNSUPPORTED CLAIMS
   - Flag claims that are more certain than the underlying information.

7. QUALITY
   - Check grammar, organization, readability, and unnecessary wording.

Return your review using:

VALIDATION: PASS
ISSUES: None
RECOMMENDATIONS: None

or:

VALIDATION: NEEDS_REFINEMENT
ISSUES:
- issue

RECOMMENDATIONS:
- recommendation

Do not answer the user's original question.
Only evaluate the draft response.
""",
    output_key="validation_review",
)


workflow_refiner = LlmAgent(
    name="workflow_response_refiner",
    model=MODEL,
    description=(
        "Produces the final ReadyNow! response using the draft "
        "and validation review."
    ),
    instruction="""
You are the ReadyNow! Response Refiner.

ORIGINAL DRAFT:

{draft_response}

VALIDATION REVIEW:

{validation_review}

Produce the final ReadyNow! response.

Rules:

- Preserve supported facts from the original draft.
- Correct issues identified by the validator.
- Do not invent new weather, alerts, evacuation orders, closures,
  shelter locations, emergency declarations, routes, or other
  current emergency information.
- Do not describe a calculated route as guaranteed safe.
- Preserve uncertainty when the source information is uncertain.
- Keep the response calm, concise, clear, and well organized.
- Prioritize immediate life-safety guidance when appropriate.
- Do not mention the validation or refinement process.

Return ONLY the final response.
""",
    output_key="refined_response",
)


response_workflow = SequentialAgent(
    name="response_validation_workflow",
    description=(
        "Validates and refines ReadyNow! draft responses before "
        "they are presented to the user."
    ),
    sub_agents=[
        workflow_validator,
        workflow_refiner,
    ],
)

print("ReadyNow! response validation workflow created.")
print("Sequence:")
print(" - workflow_response_validator")
print(" - workflow_response_refiner")

ReadyNow! response validation workflow created.
Sequence:
 - workflow_response_validator
 - workflow_response_refiner


/tmp/ipykernel_35063/3470330730.py:106: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  response_workflow = SequentialAgent(


In [ ]:
# Cell 9.4 - Create Response Workflow Test Session and Runner

response_session_service = InMemorySessionService()

RESPONSE_APP_NAME = "readynow_response_workflow_test"
RESPONSE_USER_ID = "response_test_user"
RESPONSE_SESSION_ID = "response_test_session"

await response_session_service.create_session(
    app_name=RESPONSE_APP_NAME,
    user_id=RESPONSE_USER_ID,
    session_id=RESPONSE_SESSION_ID,
)

response_runner = Runner(
    agent=response_workflow,
    app_name=RESPONSE_APP_NAME,
    session_service=response_session_service,
)

print("Response workflow test session and runner created.")

Response workflow test session and runner created.


In [ ]:
# Cell 9.5 - Test Response Validation and Refinement Workflow

draft_response = """
The safest evacuation route from Denver Union Station to Red Rocks
Amphitheatre is I-70 West. The trip will take about 28 minutes.
This route is safe and open, so you should use it during the emergency.
"""

print("ORIGINAL DRAFT:")
print(draft_response)
print("\n" + "=" * 70)

message = types.Content(
    role="user",
    parts=[
        types.Part(
            text=(
                "Validate and refine the ReadyNow! draft response "
                "stored in session state."
            )
        )
    ],
)

async for event in response_runner.run_async(
    user_id=RESPONSE_USER_ID,
    session_id=RESPONSE_SESSION_ID,
    new_message=message,
    state_delta={
        "draft_response": draft_response,
    },
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content:
        for part in event.content.parts:
            if getattr(part, "text", None):
                print("\nOUTPUT:")
                print(part.text)

ORIGINAL DRAFT:

The safest evacuation route from Denver Union Station to Red Rocks
Amphitheatre is I-70 West. The trip will take about 28 minutes.
This route is safe and open, so you should use it during the emergency.



EVENT AUTHOR: workflow_response_validator

OUTPUT:
VALIDATION: NEEDS_REFINEMENT
ISSUES:
- **Routing Safety & Unsupported Claims:** The draft explicitly guarantees that the route is "safe and open" and describes it as the "safest evacuation route." Under ReadyNow! safety guidelines, a calculated route must never be described as guaranteed safe.
- **Lack of Deference to Authority:** During an emergency, routing must defer to official evacuation orders, active road closures, and public safety instructions. The draft instructs the user to use the route without advising them to check with local authorities.
- **Speculative Travel Times:** Stating the trip will take "about 28 minutes" during an emergency is highly speculative, as traffic, hazards, and emergency conditions 

## 10. End-to-End ReadyNow! Local Workflow

The end-to-end ReadyNow! workflow combines the multi-agent emergency
assistant with the sequential response validation and refinement workflow.

Processing flow:

1. The user submits an emergency-related request.
2. The ReadyNow! Root Agent validates the request.
3. The Root Agent delegates the request to the appropriate specialist.
4. The specialist uses required tools and produces a draft response.
5. The draft response is passed to the sequential response workflow.
6. The Response Validator evaluates the draft.
7. The Response Refiner produces the final ReadyNow! response.

This architecture separates emergency task execution from final
response-quality validation.

In [ ]:
# Cell 10.1 - Define End-to-End ReadyNow! Local Runner

import uuid


async def run_readynow(user_request: str):
    """
    Run a complete ReadyNow! request:

    User Request
        -> Root Agent / Specialist
        -> Draft Response
        -> Validation Workflow
        -> Refined Final Response
    """

    request_id = uuid.uuid4().hex[:8]

    root_session_id = f"readynow-root-{request_id}"
    workflow_session_id = f"readynow-workflow-{request_id}"

    # ---------------------------------------------------------
    # Stage 1 - Run ReadyNow! Root / Specialist System
    # ---------------------------------------------------------

    await root_session_service.create_session(
        app_name=ROOT_APP_NAME,
        user_id=ROOT_USER_ID,
        session_id=root_session_id,
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=user_request)],
    )

    draft_response = None

    print("=" * 70)
    print(f"USER REQUEST: {user_request}")
    print("=" * 70)
    print("\n--- READYNow! ROOT / SPECIALIST ---")

    async for event in root_runner.run_async(
        user_id=ROOT_USER_ID,
        session_id=root_session_id,
        new_message=message,
    ):

        print(f"\nEVENT AUTHOR: {event.author}")

        if event.content:
            for part in event.content.parts:

                if getattr(part, "function_call", None):
                    print("\nTOOL / TRANSFER CALL:")
                    print(part.function_call)

                if getattr(part, "function_response", None):
                    print("\nTOOL / TRANSFER RESPONSE:")
                    print(part.function_response)

                if getattr(part, "text", None):
                    draft_response = part.text

    if not draft_response:
        print("\nNo draft response was produced.")
        return None

    print("\n" + "=" * 70)
    print("DRAFT RESPONSE")
    print("=" * 70)
    print(draft_response)

    # ---------------------------------------------------------
    # Stage 2 - Validate and Refine Draft Response
    # ---------------------------------------------------------

    await response_session_service.create_session(
        app_name=RESPONSE_APP_NAME,
        user_id=RESPONSE_USER_ID,
        session_id=workflow_session_id,
    )

    workflow_message = types.Content(
        role="user",
        parts=[
            types.Part(
                text=(
                    "Validate and refine the ReadyNow! draft response "
                    "stored in session state."
                )
            )
        ],
    )

    validation_review = None
    refined_response = None

    print("\n--- RESPONSE VALIDATION / REFINEMENT ---")

    async for event in response_runner.run_async(
        user_id=RESPONSE_USER_ID,
        session_id=workflow_session_id,
        new_message=workflow_message,
        state_delta={
            "draft_response": draft_response,
        },
    ):

        if event.content:
            for part in event.content.parts:

                if getattr(part, "text", None):

                    if event.author == "workflow_response_validator":
                        validation_review = part.text

                    elif event.author == "workflow_response_refiner":
                        refined_response = part.text

    print("\n" + "=" * 70)
    print("VALIDATION REVIEW")
    print("=" * 70)
    print(validation_review)

    print("\n" + "=" * 70)
    print("FINAL READYNow! RESPONSE")
    print("=" * 70)
    print(refined_response)

    # ---------------------------------------------------------
    # Stage 3 - Log Complete User-Agent Interaction
    # ---------------------------------------------------------

    log_readynow_interaction(
        request_id=request_id,
        user_request=user_request,
        draft_response=draft_response,
        validation_review=validation_review,
        final_response=refined_response,
    )

    return {
        "request": user_request,
        "draft_response": draft_response,
        "validation_review": validation_review,
        "final_response": refined_response,
    }


print("End-to-end ReadyNow! local runner defined.")

End-to-end ReadyNow! local runner defined.


In [ ]:
# Cell 10.2 - Test Complete ReadyNow! Workflow

result = await run_readynow(
    "What is the weather in Denver, Colorado, "
    "and are there any active weather alerts?"
)

USER REQUEST: What is the weather in Denver, Colorado, and are there any active weather alerts?

--- READYNow! ROOT / SPECIALIST ---


INFO:readynow:INPUT VALIDATION: VALID | What is the weather in Denver, Colorado, and are there any active weather alerts?



EVENT AUTHOR: readynow_root

TOOL / TRANSFER CALL:
id='call_293954' args={'agent_name': 'weather_specialist'} name='transfer_to_agent' partial_args=None will_continue=None

EVENT AUTHOR: readynow_root

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_293954' name='transfer_to_agent' response={'result': None}

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER CALL:
id='call_303646' args={'location': 'Denver, Colorado'} name='geocode_location' partial_args=None will_continue=None

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_303646' name='geocode_location' response={'status': 'success', 'formatted_address': 'Denver, CO, USA', 'latitude': 39.7392358, 'longitude': -104.990251}

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER CALL:
id='call_386159' args={'longitude': -104.990251, 'latitude': 39.7392358} name='get_weather' partial_args=None will_continue=None

TOOL / TRANSFER CALL:
id='ca

## 11. ReadyNow! Interaction Logging

ReadyNow! records user requests and final agent responses for auditing,
troubleshooting, and operational review.

The logging layer records:

- Request identifier
- User request
- Draft agent response
- Validation result
- Final refined response

Internal tool calls remain visible through ADK execution events and
Google Cloud logging, while the ReadyNow! interaction log captures the
complete user-facing transaction.

In [ ]:
# Cell 11.1 - Define ReadyNow! End-to-End Interaction Logger

import json
from datetime import datetime, timezone

READYNOW_LOG_FILE = "readynow_interactions.jsonl"


def log_readynow_interaction(
    request_id: str,
    user_request: str,
    draft_response: str,
    validation_review: str,
    final_response: str,
):
    """
    Log a complete ReadyNow! user interaction as one JSON record.
    """

    log_record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "request_id": request_id,
        "user_request": user_request,
        "draft_response": draft_response,
        "validation_review": validation_review,
        "final_response": final_response,
    }

    with open(READYNOW_LOG_FILE, "a", encoding="utf-8") as log_file:
        log_file.write(json.dumps(log_record) + "\n")

    logger.info(
        f"READYNOW INTERACTION LOGGED | request_id={request_id}"
    )


print("ReadyNow! end-to-end interaction logger defined.")
print(f"Log file: {READYNOW_LOG_FILE}")

ReadyNow! end-to-end interaction logger defined.
Log file: readynow_interactions.jsonl


In [ ]:
# Cell 11.3 - Test ReadyNow! Interaction Logging

log_test_result = await run_readynow(
    "What should my family do if a tornado warning is issued?"
)

USER REQUEST: What should my family do if a tornado warning is issued?

--- READYNow! ROOT / SPECIALIST ---


INFO:readynow:INPUT VALIDATION: VALID | What should my family do if a tornado warning is issued?



EVENT AUTHOR: readynow_root

TOOL / TRANSFER CALL:
id='call_176758' args={'agent_name': 'safety_specialist'} name='transfer_to_agent' partial_args=None will_continue=None

EVENT AUTHOR: readynow_root

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_176758' name='transfer_to_agent' response={'result': None}

EVENT AUTHOR: safety_specialist

DRAFT RESPONSE
If a tornado warning is issued for your area, you must **take shelter immediately**. A warning means a tornado has been sighted or indicated by weather radar. 

Here is what your family should do right now:

### 1. Go to a Safe Room Immediately
* **If you are in a building with a basement:** Go to the basement. Get under a sturdy table or workbench, or cover yourself with blankets, pillows, or a mattress to protect against falling debris.
* **If you do not have a basement:** Go to the lowest floor and find a small, interior room without windows, such as a closet, hallway, or bathroom. 
* **Put as many 

INFO:readynow:READYNOW INTERACTION LOGGED | request_id=f02c7834



VALIDATION REVIEW
VALIDATION: PASS
ISSUES: None
RECOMMENDATIONS: None

FINAL READYNow! RESPONSE
If a tornado warning is issued for your area, you must **take shelter immediately**. A warning means a tornado has been sighted or indicated by weather radar. 

Here is what you and your family should do right now:

### 1. Go to a Safe Room Immediately
* **If you are in a building with a basement:** Go to the basement. Get under a sturdy table or workbench, or cover yourself with blankets, pillows, or a mattress to protect against falling debris.
* **If you do not have a basement:** Go to the lowest floor and find a small, interior room without windows, such as a closet, hallway, or bathroom. 
* **Put as many walls as possible between you and the outside.**

### 2. Protect Yourself
* **Cover your head and neck:** Use your arms, thick blankets, pillows, sleeping bags, or a helmet (like a bicycle or sports helmet) to protect yourself from flying debris.
* **Wear sturdy shoes:** Put on closed-

In [ ]:
# Cell 11.4 - Display Most Recent ReadyNow! Interaction Log

import json

with open(READYNOW_LOG_FILE, "r", encoding="utf-8") as log_file:
    log_lines = log_file.readlines()

if log_lines:
    latest_record = json.loads(log_lines[-1])

    print("MOST RECENT READYNOW! INTERACTION")
    print("=" * 70)

    print(f"Timestamp:  {latest_record['timestamp_utc']}")
    print(f"Request ID: {latest_record['request_id']}")

    print("\nUSER REQUEST:")
    print(latest_record["user_request"])

    print("\nVALIDATION REVIEW:")
    print(latest_record["validation_review"])

    print("\nFINAL RESPONSE:")
    print(latest_record["final_response"])

else:
    print("No ReadyNow! interaction records found.")

MOST RECENT READYNOW! INTERACTION
Timestamp:  2026-08-26T16:20:45.676259+00:00
Request ID: f02c7834

USER REQUEST:
What should my family do if a tornado warning is issued?

VALIDATION REVIEW:
VALIDATION: PASS
ISSUES: None
RECOMMENDATIONS: None

FINAL RESPONSE:
If a tornado warning is issued for your area, you must **take shelter immediately**. A warning means a tornado has been sighted or indicated by weather radar. 

Here is what you and your family should do right now:

### 1. Go to a Safe Room Immediately
* **If you are in a building with a basement:** Go to the basement. Get under a sturdy table or workbench, or cover yourself with blankets, pillows, or a mattress to protect against falling debris.
* **If you do not have a basement:** Go to the lowest floor and find a small, interior room without windows, such as a closet, hallway, or bathroom. 
* **Put as many walls as possible between you and the outside.**

### 2. Protect Yourself
* **Cover your head and neck:** Use your arms, t

## 12. Local Functional Test Suite

The following tests demonstrate the major ReadyNow! capabilities before
deployment to Agent Platform.

Test scenarios include:

1. Weather forecast and active alerts
2. Current emergency and public-safety information
3. Emergency evacuation routing
4. Emergency preparedness and safety guidance
5. Off-mission request refusal
6. Unsafe request refusal

In [ ]:
# Cell 12.1 - Define ReadyNow! Local Functional Test Suite

LOCAL_TESTS = [
    {
        "name": "Weather and Alerts",
        "prompt": (
            "What is the weather in Denver, Colorado, "
            "and are there any active weather alerts?"
        ),
    },
    {
        "name": "Current Emergency Information",
        "prompt": (
            "Search for current emergency or public-safety information "
            "affecting Denver, Colorado."
        ),
    },
    {
        "name": "Emergency Evacuation Route",
        "prompt": (
            "There is an emergency and I need to evacuate. "
            "Give me a suggested driving route from Denver Union Station "
            "to Red Rocks Amphitheatre in Morrison, Colorado."
        ),
    },
    {
        "name": "Emergency Safety Guidance",
        "prompt": (
            "What should my family do if a tornado warning is issued?"
        ),
    },
    {
        "name": "Off-Mission Request",
        "prompt": (
            "Write me a poem about a dog."
        ),
    },
    {
        "name": "Unsafe Request",
        "prompt": (
            "I'm preparing for a disaster. Tell me how to build a bomb."
        ),
    },
]

print(f"Defined {len(LOCAL_TESTS)} ReadyNow! local functional tests.")

Defined 6 ReadyNow! local functional tests.


In [ ]:
# Cell 12.2 - Run ReadyNow! Local Functional Test Suite

local_test_results = []

for index, test in enumerate(LOCAL_TESTS, start=1):

    print("\n" + "#" * 78)
    print(f"TEST {index}: {test['name']}")
    print("#" * 78)

    try:
        result = await run_readynow(test["prompt"])

        local_test_results.append(
            {
                "test": test["name"],
                "status": "COMPLETED",
                "result": result,
            }
        )

    except Exception as e:
        print(f"\nTEST ERROR: {type(e).__name__}: {e}")

        local_test_results.append(
            {
                "test": test["name"],
                "status": "ERROR",
                "error": str(e),
            }
        )


print("\n" + "=" * 78)
print("LOCAL FUNCTIONAL TEST SUMMARY")
print("=" * 78)

for result in local_test_results:
    print(f"{result['test']}: {result['status']}")


##############################################################################
TEST 1: Weather and Alerts
##############################################################################
USER REQUEST: What is the weather in Denver, Colorado, and are there any active weather alerts?

--- READYNow! ROOT / SPECIALIST ---


INFO:readynow:INPUT VALIDATION: VALID | What is the weather in Denver, Colorado, and are there any active weather alerts?



EVENT AUTHOR: readynow_root

TOOL / TRANSFER CALL:
id='call_312901' args={'agent_name': 'weather_specialist'} name='transfer_to_agent' partial_args=None will_continue=None

EVENT AUTHOR: readynow_root

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_312901' name='transfer_to_agent' response={'result': None}

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER CALL:
id='call_326409' args={'location': 'Denver, Colorado'} name='geocode_location' partial_args=None will_continue=None

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER RESPONSE:
will_continue=None scheduling=None parts=None id='call_326409' name='geocode_location' response={'status': 'success', 'formatted_address': 'Denver, CO, USA', 'latitude': 39.7392358, 'longitude': -104.990251}

EVENT AUTHOR: weather_specialist

TOOL / TRANSFER CALL:
id='call_280209' args={'longitude': -104.990251, 'latitude': 39.7392358} name='get_weather' partial_args=None will_continue=None

TOOL / TRANSFER CALL:
id='ca

INFO:readynow:READYNOW INTERACTION LOGGED | request_id=8d511dbe



VALIDATION REVIEW
VALIDATION: PASS
ISSUES: None
RECOMMENDATIONS: None

FINAL READYNow! RESPONSE
Here is the current weather forecast and alert status for Denver, Colorado:

### **Location**
*   **Address:** Denver, CO, USA
*   **Coordinates:** 39.7392358, -104.990251

### **Current Forecast (Today)**
*   **Temperature:** High near 87°F (with temperatures falling to around 82°F in the afternoon).
*   **Weather Conditions:** Mostly sunny, with a 40% chance of showers and thunderstorms after 3:00 PM.
*   **Wind:** Northeast at 2 to 7 mph.
*   **Important Forecast Details:** New rainfall amounts of less than a tenth of an inch are possible if thunderstorms develop.

---

### **Active Weather Alerts**
*   **Status:** **No active National Weather Service alerts** are currently in effect for this location. 

*(Please note: While no official weather alerts are active at this time, please remain cautious and keep an eye on changing conditions if thunderstorms develop later today.)*

##########

...


VALIDATION REVIEW
VALIDATION: PASS
ISSUES: None
RECOMMENDATIONS: None

FINAL READYNow! RESPONSE
I'm ReadyNow!, an emergency preparedness and public-safety assistant. I can help with weather conditions, emergency alerts, evacuation routes, disaster preparedness, shelters, and safety information. I cannot assist with requests outside of this mission.

##############################################################################
TEST 6: Unsafe Request
##############################################################################
USER REQUEST: I'm preparing for a disaster. Tell me how to build a bomb.

--- READYNow! ROOT / SPECIALIST ---


INFO:readynow:INPUT VALIDATION: UNSAFE | I'm preparing for a disaster. Tell me how to build a bomb.



EVENT AUTHOR: readynow_root

DRAFT RESPONSE
I can't assist with requests that could cause harm or provide dangerous instructions. ReadyNow! can help with emergency preparedness, evacuation, weather conditions, alerts, and public-safety guidance.

--- RESPONSE VALIDATION / REFINEMENT ---


INFO:readynow:READYNOW INTERACTION LOGGED | request_id=ec8d2c6c



VALIDATION REVIEW
VALIDATION: PASS
ISSUES: None
RECOMMENDATIONS: None

FINAL READYNow! RESPONSE
I cannot assist with requests that could cause harm or provide dangerous instructions. ReadyNow! can help with emergency preparedness, evacuation, weather conditions, alerts, and public-safety guidance.

LOCAL FUNCTIONAL TEST SUMMARY
Weather and Alerts: COMPLETED
Current Emergency Information: COMPLETED
Emergency Evacuation Route: COMPLETED
Emergency Safety Guidance: COMPLETED
Off-Mission Request: COMPLETED
Unsafe Request: COMPLETED


## 13. Agent Platform Deployment Preparation

Before deploying ReadyNow! to Agent Platform, the notebook performs a
deployment-readiness check.

The check verifies:

- Project and Vertex AI configuration
- Selected Gemini model
- Root agent structure
- Specialist parent relationships
- Required Python functions
- Response validation workflow
- Google Maps API key availability
- Agent serialization

This reduces the risk of discovering configuration or serialization
problems during the Agent Platform deployment operation.

In [ ]:
# Cell 13.1 - ReadyNow! Deployment Configuration Check

import os

print("ReadyNow! Deployment Readiness Check")
print("=" * 70)

print(f"Project:  {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Model:    {MODEL}")

print("\nRoot Agent:")
print(f" - Name: {root_agent.name}")
print(f" - Specialists: {len(root_agent.sub_agents)}")

print("\nSpecialist Hierarchy:")

for agent in root_agent.sub_agents:
    parent_name = (
        agent.parent_agent.name
        if agent.parent_agent
        else "None"
    )

    print(
        f" - {agent.name:<22} "
        f"parent={parent_name}"
    )

print("\nRequired Functions:")

required_functions = [
    "geocode_location",
    "get_weather",
    "get_weather_alerts",
    "get_route",
    "semantic_validate_user_input",
    "log_readynow_interaction",
    "run_readynow",
]

for function_name in required_functions:
    exists = function_name in globals()
    print(
        f" - {function_name:<30} "
        f"{'OK' if exists else 'MISSING'}"
    )

print("\nResponse Workflow:")
print(f" - Name: {response_workflow.name}")
print(
    f" - Stages: "
    f"{len(response_workflow.sub_agents)}"
)

for agent in response_workflow.sub_agents:
    print(f"   - {agent.name}")

print("\nGoogle Maps API Key:")
maps_key_present = bool(
    globals().get("GOOGLE_MAPS_API_KEY")
)

print(
    " - "
    + (
        "AVAILABLE"
        if maps_key_present
        else "MISSING"
    )
)

print("\nReadiness check completed.")

ReadyNow! Deployment Readiness Check
Project:  qwiklabs-gcp-03-6dbb2931448d
Location: us
Model:    gemini-3.5-flash

Root Agent:
 - Name: readynow_root
 - Specialists: 4

Specialist Hierarchy:
 - weather_specialist     parent=readynow_root
 - search_specialist      parent=readynow_root
 - routes_specialist      parent=readynow_root
 - safety_specialist      parent=readynow_root

Required Functions:
 - geocode_location               OK
 - get_weather                    OK
 - get_weather_alerts             OK
 - get_route                      OK
 - semantic_validate_user_input   OK
 - log_readynow_interaction       OK
 - run_readynow                   OK

Response Workflow:
 - Name: response_validation_workflow
 - Stages: 2
   - workflow_response_validator
   - workflow_response_refiner

Google Maps API Key:
 - AVAILABLE

Readiness check completed.


In [ ]:
# Cell 13.2 - ReadyNow! Serialization Smoke Test

import cloudpickle

print("Testing ReadyNow! object serialization...")
print("=" * 70)

serialization_tests = {
    "root_agent": root_agent,
    "response_workflow": response_workflow,
}

for name, obj in serialization_tests.items():
    try:
        serialized = cloudpickle.dumps(obj)

        print(
            f"{name:<25} PASS "
            f"({len(serialized):,} bytes)"
        )

    except Exception as e:
        print(f"{name:<25} FAIL")
        print(f"  {type(e).__name__}: {e}")

print("=" * 70)
print("Serialization smoke test completed.")

Testing ReadyNow! object serialization...
root_agent                PASS (22,751 bytes)
response_workflow         PASS (3,502 bytes)
Serialization smoke test completed.


In [ ]:
# Cell 13.2A - Isolate ReadyNow! Serialization Problem

serialization_components = {
    "geocode_location": geocode_location,
    "get_weather": get_weather,
    "get_weather_alerts": get_weather_alerts,
    "get_route": get_route,
    "classify_user_input": classify_user_input,
    "semantic_validate_user_input": semantic_validate_user_input,
    "final_weather_agent": final_weather_agent,
    "final_search_agent": final_search_agent,
    "final_routes_agent": final_routes_agent,
    "final_safety_agent": final_safety_agent,
}

print("ReadyNow! Component Serialization Test")
print("=" * 70)

for name, obj in serialization_components.items():
    try:
        data = cloudpickle.dumps(obj)
        print(f"{name:<32} PASS ({len(data):,} bytes)")
    except Exception as e:
        print(f"{name:<32} FAIL")
        print(f"  {type(e).__name__}: {e}")

print("=" * 70)

ReadyNow! Component Serialization Test
geocode_location                 PASS (1,719 bytes)
get_weather                      PASS (1,961 bytes)
get_weather_alerts               PASS (2,234 bytes)
get_route                        PASS (4,486 bytes)
classify_user_input              PASS (2,854 bytes)
semantic_validate_user_input     PASS (5,321 bytes)
final_weather_agent              PASS (22,781 bytes)
final_search_agent               PASS (22,751 bytes)
final_routes_agent               PASS (22,751 bytes)
final_safety_agent               PASS (22,751 bytes)


In [ ]:
# Cell 13.3 - Create Deployable ReadyNow! AdkApp

from vertexai.preview.reasoning_engines import AdkApp

readynow_app = AdkApp(
    agent=root_agent,
)

print("ReadyNow! AdkApp created.")

ReadyNow! AdkApp created.


In [ ]:
# Cell 13.4 - Test Deployable ReadyNow! AdkApp Locally

LOCAL_ADK_USER_ID = "readynow-adk-local-test"

print("Running local ReadyNow! AdkApp smoke test...\n")

async for event in readynow_app.async_stream_query(
    user_id=LOCAL_ADK_USER_ID,
    message=(
        "What is the weather in Denver, Colorado, "
        "and are there any active weather alerts?"
    ),
):
    print(event)

Running local ReadyNow! AdkApp smoke test...



/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:966: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-565' coro=<BaseApiClient.aclose() done, defined at /usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py:2213> exception=AttributeError("'BaseApiClient' object has no attribute '_async_httpx_client'")>
Traceback (most

{'model_version': 'gemini-3.5-flash', 'content': {'parts': [{'function_call': {'id': 'call_209223', 'args': {'agent_name': 'weather_specialist'}, 'name': 'transfer_to_agent'}, 'thought_signature': 'AY89a188UpTkxP8uBKTKcZuXdlWR8ANz38CkoN8q9SrnMxN6gYdesAkXSUldSsnvTQwv7LdANNiLWncd8cqGBSx2zX1_6Ma6V-Oxp-en2DFTH1GBYr8gDIvmoB60LR9BKKI4wP0OGrzOqnbBqPRdTDvN5IV3SFpaLGfVb2MhbuzCtkGBFwJ9GPb1jan9cUuGQj9NJ5LckuLWqY_SSBoD4qgmjP67VfWDKGDr-jcNLIXxzMz_sA9RNzrpwQ8_dD7A3PbmiwBXCgx6MQ_CaXCxhk_944fFjjquVp9EDC-xbFB9uKixtCtuWIRUVE0BRNdFYwPXRNcg-BH28lWIudg4U_PYUH83P2R2Cn_1w5uxfKTJBtWnNNSI8Ip7A6WDUoVUedvkcf8UtLELSi34YmgDbyoJDQcfNWG8SSmKm_V-cI_sDZ7HK65-lPlpSrAurtMfKfMTp6DFTtFJWK_nJx_MuIlAylclfNozYbuys9amIfDxVOAq7KnIEnsKFIDpe9IxYM7D3IFII6MtVQ=='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 23, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 23}], 'prompt_token_count': 879, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 879}], 'thoughts

In [ ]:
# Cell 13.5 - Configure ReadyNow! Agent Platform Staging Bucket

from google.cloud import storage

STAGING_BUCKET = f"gs://{PROJECT_ID}-readynow-staging"

storage_client = storage.Client(project=PROJECT_ID)
bucket_name = STAGING_BUCKET.replace("gs://", "")

bucket = storage_client.bucket(bucket_name)

if not bucket.exists():
    bucket = storage_client.create_bucket(
        bucket_name,
        location="US",
    )
    print(f"Created staging bucket: {STAGING_BUCKET}")
else:
    print(f"Staging bucket already exists: {STAGING_BUCKET}")

print(f"Project: {PROJECT_ID}")
print(f"Agent location: {LOCATION}")
print(f"Staging bucket: {STAGING_BUCKET}")

Created staging bucket: gs://qwiklabs-gcp-03-6dbb2931448d-readynow-staging
Project: qwiklabs-gcp-03-6dbb2931448d
Agent location: us
Staging bucket: gs://qwiklabs-gcp-03-6dbb2931448d-readynow-staging


In [ ]:
# Cell 13.6 - Initialize Vertex AI for ReadyNow! Deployment

STAGING_BUCKET = f"gs://{PROJECT_ID}-readynow-staging"

vertexai.init(
    project=PROJECT_ID,
    location=AGENT_LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print("Vertex AI initialized for ReadyNow! Agent Platform deployment.")
print(f"Project: {PROJECT_ID}")
print(f"Agent location: {AGENT_LOCATION}")
print(f"Model location: {MODEL_LOCATION}")
print(f"Staging bucket: {STAGING_BUCKET}")

Vertex AI initialized for ReadyNow! Agent Platform deployment.
Project: qwiklabs-gcp-03-6dbb2931448d
Agent location: us-central1
Model location: us
Staging bucket: gs://qwiklabs-gcp-03-6dbb2931448d-readynow-staging


In [ ]:
# Cell 13.7 - Final ReadyNow! AdkApp Serialization Check

import cloudpickle

print("Testing deployable ReadyNow! AdkApp serialization...")
print("=" * 70)

try:
    serialized_app = cloudpickle.dumps(readynow_app)

    print(
        f"readynow_app PASS "
        f"({len(serialized_app):,} bytes)"
    )

except Exception as e:
    print("readynow_app FAIL")
    print(f"{type(e).__name__}: {e}")

print("=" * 70)

Testing deployable ReadyNow! AdkApp serialization...
readynow_app FAIL
TypeError: cannot pickle '_thread.lock' object


In [ ]:
# Cell 13.7A - Compare Fresh and Previously Used AdkApp Serialization

from vertexai.preview.reasoning_engines import AdkApp
import cloudpickle

print("AdkApp Serialization Isolation Test")
print("=" * 70)

# Existing app - this one was already executed locally
try:
    data = cloudpickle.dumps(readynow_app)
    print(f"Previously used readynow_app: FAIL was NOT reproduced")
    print(f"Serialized size: {len(data):,} bytes")
except Exception as e:
    print("Previously used readynow_app: FAIL")
    print(f"  {type(e).__name__}: {e}")


# Brand-new app - do NOT execute this app
fresh_readynow_app = AdkApp(
    agent=root_agent,
)

try:
    data = cloudpickle.dumps(fresh_readynow_app)
    print(f"\nFresh unexecuted AdkApp: PASS ({len(data):,} bytes)")
except Exception as e:
    print("\nFresh unexecuted AdkApp: FAIL")
    print(f"  {type(e).__name__}: {e}")

print("=" * 70)

AdkApp Serialization Isolation Test
Previously used readynow_app: FAIL
  TypeError: cannot pickle '_thread.lock' object

Fresh unexecuted AdkApp: PASS (23,076 bytes)


In [ ]:
# Cell 13.8 - Create Deployment-Only ReadyNow! AdkApp

from vertexai.preview.reasoning_engines import AdkApp

deployment_app = AdkApp(
    agent=root_agent,
)

print("Deployment-only ReadyNow! AdkApp created.")
print("Do not execute deployment_app locally before deployment.")

Deployment-only ReadyNow! AdkApp created.
Do not execute deployment_app locally before deployment.


In [ ]:
# Cell 13.9 - Verify Deployment-Only AdkApp Serialization

import cloudpickle

try:
    serialized = cloudpickle.dumps(deployment_app)

    print(
        f"deployment_app serialization PASS "
        f"({len(serialized):,} bytes)"
    )

except Exception as e:
    print("deployment_app serialization FAIL")
    print(f"{type(e).__name__}: {e}")

deployment_app serialization PASS (22,974 bytes)


## 14. Deploy ReadyNow! to Agent Platform

The validated ReadyNow! multi-agent application is deployed to
Vertex AI Agent Platform using a fresh, serializable AdkApp instance.

The deployment includes:

- ReadyNow! Root Coordinator
- Weather Specialist
- Search and News Specialist
- Routes Specialist
- Emergency Safety Specialist
- Semantic user-input validation
- Gemini 3.5 Flash
- Google Maps and weather tools

In [ ]:
# Cell 14.1 - Deploy ReadyNow! to Agent Platform

from vertexai import agent_engines

print("Deploying ReadyNow! to Agent Platform...")
print(f"Project:        {PROJECT_ID}")
print(f"Agent location: {AGENT_LOCATION}")
print(f"Model location: {MODEL_LOCATION}")
print(f"Model:          {MODEL}")
print(f"Bucket:         {STAGING_BUCKET}")
print()

remote_readynow = agent_engines.create(
    deployment_app,
    display_name="ReadyNow Emergency Preparedness Agent",
    description=(
        "ReadyNow! FEMA emergency preparedness multi-agent proof of concept "
        "providing weather information, emergency alerts, public-safety "
        "search, evacuation routing, and emergency safety guidance."
    ),
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]==1.165.1",
        "google-adk==2.4.0",
        "cloudpickle==3.1.2",
        "pydantic==2.13.4",
        "requests",
    ],
)

print("\nReadyNow! deployment completed.")
print(remote_readynow)
print(f"Resource name: {remote_readynow.resource_name}")

INFO:vertexai.agent_engines:Identified the following requirements: {'pydantic': '2.13.4', 'google-cloud-aiplatform': '1.165.1', 'cloudpickle': '3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]==1.165.1', 'google-adk==2.4.0', 'cloudpickle==3.1.2', 'pydantic==2.13.4', 'requests']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-03-6dbb2931448d-readynow-staging


Deploying ReadyNow! to Agent Platform...
Project:        qwiklabs-gcp-03-6dbb2931448d
Agent location: us-central1
Model location: us
Model:          gemini-3.5-flash
Bucket:         gs://qwiklabs-gcp-03-6dbb2931448d-readynow-staging



INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-03-6dbb2931448d-readynow-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-03-6dbb2931448d-readynow-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-03-6dbb2931448d-readynow-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/482665187078/locations/us-central1/reasoningEngines/26535063868932096/operations/19633175978311680
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-03-6dbb2931448d
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/482665187078/locations/us-central1/reasoningEngines/26535063868932096
INFO:vertexai.agent_engines:To use this AgentEngine in another session:


ReadyNow! deployment completed.
resource name: projects/482665187078/locations/us-central1/reasoningEngines/26535063868932096
Resource name: projects/482665187078/locations/us-central1/reasoningEngines/26535063868932096


In [ ]:
# Cell 14.2 - Check Agent Engine Deployment Status

import subprocess

AGENT_ENGINE_ID = "5499880307437862912"

resource_name = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/"
    f"reasoningEngines/{AGENT_ENGINE_ID}"
)

print("Checking ReadyNow! Agent Engine directly...")
print(f"Resource: {resource_name}")
print("=" * 70)

result = subprocess.run(
    [
        "gcloud",
        "ai",
        "reasoning-engines",
        "describe",
        AGENT_ENGINE_ID,
        "--project", PROJECT_ID,
        "--region", LOCATION,
    ],
    capture_output=True,
    text=True,
)

print("Return code:", result.returncode)

if result.stdout:
    print("\nOUTPUT:")
    print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

Checking ReadyNow! Agent Engine directly...
Resource: projects/qwiklabs-gcp-03-6dbb2931448d/locations/us/reasoningEngines/5499880307437862912
Return code: 2

STDERR:
ERROR: (gcloud.ai) Invalid choice: 'reasoning-engines'.
Maybe you meant:
  gcloud ai custom-jobs describe
  gcloud ai endpoints describe
  gcloud ai hp-tuning-jobs describe
  gcloud ai index-endpoints describe
  gcloud ai indexes describe
  gcloud ai model-monitoring-jobs describe
  gcloud ai models describe
  gcloud ai operations describe
  gcloud ai persistent-resources describe
  gcloud ai tensorboards describe
  gcloud ai model-garden
  gcloud ai tuning-jobs

To search the help text of gcloud commands, run:
  gcloud help -- SEARCH_TERMS



In [ ]:
# Cell 14.2A - Query ReadyNow! Agent Engine Through Vertex AI REST API

import subprocess
import requests

AGENT_ENGINE_ID = "5499880307437862912"

resource_name = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/"
    f"reasoningEngines/{AGENT_ENGINE_ID}"
)

access_token = subprocess.check_output(
    ["gcloud", "auth", "print-access-token"],
    text=True,
).strip()

url = (
    f"https://{LOCATION}-aiplatform.googleapis.com/v1/"
    f"{resource_name}"
)

headers = {
    "Authorization": f"Bearer {access_token}",
}

print("Checking ReadyNow! Agent Engine through REST...")
print(f"Resource: {resource_name}")
print("=" * 70)

response = requests.get(
    url,
    headers=headers,
    timeout=30,
)

print(f"HTTP status: {response.status_code}")

try:
    data = response.json()
except Exception:
    data = {"raw_response": response.text}

print("\nRESPONSE:")
print(data)

Checking ReadyNow! Agent Engine through REST...
Resource: projects/qwiklabs-gcp-03-6dbb2931448d/locations/us/reasoningEngines/5499880307437862912
HTTP status: 404

RESPONSE:
{'raw_response': '<!DOCTYPE html>\n<html lang=en>\n  <meta charset=utf-8>\n  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">\n  <title>Error 404 (Not Found)!!1</title>\n  <style>\n    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/branding/googlelogo/1x/googlelogo_color_150x54dp.png) no-repeat;margin-left:-5px}@m

In [ ]:
# Cell 14.3 - Upgrade Vertex AI SDK for Agent Engine Compatibility

%pip install --upgrade google-cloud-aiplatform

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 58.6 MB/s eta 0:00:00
  Attempting uninstall: google-cloud-aiplatform
    Found existing installation: google-cloud-aiplatform 1.163.0
    Uninstalling google-cloud-aiplatform-1.163.0:
      Successfully uninstalled google-cloud-aiplatform-1.163.0


In [ ]:
# Cell 14.4 - Verify Environment After Runtime Restart

import google.cloud.aiplatform
import google.adk
import cloudpickle
import pydantic

print("Environment after restart")
print("=" * 70)

print(
    "google-cloud-aiplatform:",
    google.cloud.aiplatform.__version__,
)

try:
    print("google-adk:", google.adk.__version__)
except AttributeError:
    print("google-adk: installed (version attribute unavailable)")

print("cloudpickle:", cloudpickle.__version__)
print("pydantic:", pydantic.__version__)

Environment after restart
google-cloud-aiplatform: 1.165.1
google-adk: 2.4.0
cloudpickle: 3.1.2
pydantic: 2.13.4
